# 08 — Hyperparameter Tuning, Threshold Calibration, and Evaluation

## Objective

Select model configurations using the designated development data, calibrate anomaly-score thresholds without using the final test labels, and evaluate the finalized models on the held-out test data.

This notebook separates three activities:

1. **Hyperparameter tuning** — select model configurations using the designated development data.
2. **Threshold calibration** — determine anomaly-detection thresholds from normal development data.
3. **Final evaluation** — apply the selected models and fixed thresholds to the held-out evaluation data.

The final test/attack labels are not used for hyperparameter selection or threshold calibration.

## Evaluation Principle

An anomaly-detection model produces a numerical score that represents how unusual an observation or sequence is. A threshold converts this score into a binary prediction.

Let $s_i$ denote the anomaly score for observation or sequence $i$, and let $\tau$ denote the selected threshold. The predicted label is

$$
\hat{y}_i =
\begin{cases}
1, & s_i > \tau,\\
0, & s_i \leq \tau,
\end{cases}
$$

where $\hat{y}_i = 1$ indicates a detected anomaly and $\hat{y}_i = 0$ indicates no detected anomaly.

### Reconstruction Error

For an autoencoder, reconstruction error measures the difference between the input and its reconstructed output.

Let $X_i \in \mathbb{R}^{L \times F}$ be input sequence $i$, and let $\hat{X}_i \in \mathbb{R}^{L \times F}$ be its reconstruction, where:

- $L$ is the sequence length;
- $F$ is the number of input features;
- $x_{i,t,f}$ is the observed value at timestep $t$ and feature $f$;
- $\hat{x}_{i,t,f}$ is the corresponding reconstructed value.

The mean squared reconstruction error for sequence $i$ is

$$
s_i =
\frac{1}{L F}
\sum_{t=1}^{L}
\sum_{f=1}^{F}
\left( x_{i,t,f} - \hat{x}_{i,t,f} \right)^2 .
$$

A larger value indicates a greater reconstruction discrepancy. However, a large reconstruction error is not, by itself, proof of an attack.

### Important Implementation Distinction

The equation above defines a sequence-level reconstruction score. It does not establish that every model in this project uses the same scoring procedure.

Before evaluating the models, the implementation must specify whether each method produces scores per observation, per timestep, or per sequence. For sliding windows, the relationship between sequence scores and the original timestamps must also be defined to ensure that predictions and ground-truth labels are aligned correctly.

Thresholds must be calibrated using the designated normal development data and fixed before final evaluation. Final test labels must not be used to select thresholds or tune model configurations.

## Data Separation

The project preserves the chronological structure of the SWaT time series.

Data used for:

- model training,
- hyperparameter selection,
- threshold calibration,
- and final evaluation

must remain explicitly separated.

In particular, final evaluation labels are not used to select hyperparameters or thresholds. This prevents information from the final evaluation set from influencing model decisions.

## Model Scope

The evaluation covers the anomaly-detection methods defined for this project:

### Baselines

- Rule-Based Detection
- Rolling Statistical Detection
- Simple Statistical Threshold

### Traditional Machine Learning

- Isolation Forest
- One-Class SVM
- Local Outlier Factor in novelty-detection mode

### Deep Learning

- LSTM Autoencoder
- TCN Autoencoder
- USAD

The preliminary TCN and USAD training results from Notebook 07 are treated as experimental evidence, not as final model-selection results.

## Evaluation Metrics

Performance will be assessed using appropriate classification and detection metrics, including:

- Precision
- Recall
- F1-score
- Confusion matrix
- False-positive and false-negative counts

Where appropriate, event-level behavior will also be examined in the subsequent error and event analysis stage.

No single metric will be interpreted in isolation.

In [9]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")


import torch


# Define the established Google Drive artifact directory.
PROJECT_ROOT = Path("/content/drive/MyDrive")
ARTIFACT_ROOT = PROJECT_ROOT / "swat-artifacts"
RAW_DATA_DIR = PROJECT_ROOT / "swat-dataset"

if not ARTIFACT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Artifact directory not found: {ARTIFACT_ROOT}. "
        "Verify that Google Drive is mounted in this runtime."
    )

print("SWaT ARTIFACT DIRECTORY")
print("=" * 38)
print(f"Artifact root: {ARTIFACT_ROOT}")
print(f"Raw data directory exists: {RAW_DATA_DIR.is_dir()}")

if RAW_DATA_DIR.is_dir():
    print("\nRaw data files:")
    for path in sorted(RAW_DATA_DIR.iterdir()):
        if path.is_file():
            print(f"  {path.name}")

print("\nDeep-learning artifact directories:")
for folder_name in ("tcn_preliminary", "usad_preliminary"):
    folder = ARTIFACT_ROOT / folder_name
    print(f"\n{folder_name}: {'FOUND' if folder.is_dir() else 'NOT FOUND'}")

    if folder.is_dir():
        for path in sorted(folder.iterdir()):
            if path.is_file():
                print(f"  {path.name}")

print("\nCOMPUTE ENVIRONMENT")
print("=" * 38)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU unavailable; do not start deep-learning experiments.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SWaT ARTIFACT DIRECTORY
Artifact root: /content/drive/MyDrive/swat-artifacts
Raw data directory exists: True

Raw data files:
  attack.csv
  merged.csv
  normal.csv

Deep-learning artifact directories:

tcn_preliminary: FOUND
  tcn_autoencoder_best.pth
  tcn_training_history.csv
  tcn_training_mae.png
  tcn_training_metadata.json
  tcn_training_mse.png

usad_preliminary: FOUND
  usad_best.pth
  usad_training_history.csv
  usad_training_mae.png
  usad_training_metadata.json
  usad_training_mse.png

COMPUTE ENVIRONMENT
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [10]:
from pathlib import Path

# Locate the project root using the existing project structure.
PROJECT_ROOT = Path("/content/drive/MyDrive")

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Expected a directory containing 'data/raw'. "
        "Run this notebook from within the SWaT project directory."
    )

DATA_RAW_DIR = PROJECT_ROOT / "swat-dataset"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

print(f"Raw data directory exists: {DATA_RAW_DIR.is_dir()}")
print(f"Notebooks directory exists: {NOTEBOOKS_DIR.is_dir()}")

Raw data directory exists: True
Notebooks directory exists: False


In [11]:
# Verify model artifacts in their actual project locations.

MODEL_DIR = PROJECT_ROOT / "swat-artifacts"

EXPECTED_ARTIFACTS = {
    "Isolation Forest": [
        MODEL_DIR / "isolation_forest.joblib",
    ],
    "One-Class SVM": [
        MODEL_DIR / "sgd_one_class_svm.joblib",
    ],
    "Local Outlier Factor": [
        MODEL_DIR / "lof_model.joblib",
    ],
    "LSTM Autoencoder": [
        MODEL_DIR / "lstm_autoencoder.pth",
    ],
    "TCN Autoencoder": [
        MODEL_DIR / "tcn_preliminary" / "tcn_autoencoder_best.pth",
    ],
    "USAD": [
        MODEL_DIR / "usad_preliminary" / "usad_best.pth",
    ],
    "Preprocessing": [
        MODEL_DIR / "scaler.joblib",
    ],
    "Evaluation sequences": [
        MODEL_DIR / "evaluation_sequences.npz",
    ],
}

print("NOTEBOOK 08 — MODEL ARTIFACT VERIFICATION")
print("=" * 48)

for model_name, paths in EXPECTED_ARTIFACTS.items():
    print(f"\n{model_name}:")

    for path in paths:
        if path.is_file():
            print(f"  [FOUND] {path.relative_to(PROJECT_ROOT)}")
        else:
            print(f"  [NOT FOUND] {path.relative_to(PROJECT_ROOT)}")

NOTEBOOK 08 — MODEL ARTIFACT VERIFICATION

Isolation Forest:
  [FOUND] swat-artifacts/isolation_forest.joblib

One-Class SVM:
  [FOUND] swat-artifacts/sgd_one_class_svm.joblib

Local Outlier Factor:
  [FOUND] swat-artifacts/lof_model.joblib

LSTM Autoencoder:
  [FOUND] swat-artifacts/lstm_autoencoder.pth

TCN Autoencoder:
  [FOUND] swat-artifacts/tcn_preliminary/tcn_autoencoder_best.pth

USAD:
  [FOUND] swat-artifacts/usad_preliminary/usad_best.pth

Preprocessing:
  [FOUND] swat-artifacts/scaler.joblib

Evaluation sequences:
  [FOUND] swat-artifacts/evaluation_sequences.npz


In [12]:
# Load the existing evaluation arrays once for use in Notebook 08.

import numpy as np

EVALUATION_FILE = PROJECT_ROOT / "swat-artifacts" / "evaluation_sequences.npz"

if not EVALUATION_FILE.is_file():
    raise FileNotFoundError(
        f"Evaluation file not found: {EVALUATION_FILE}"
    )

with np.load(EVALUATION_FILE, allow_pickle=False) as archive:
    EVALUATION_ARRAYS = {
        name: archive[name]
        for name in archive.files
    }

print("EVALUATION DATA READY")
print("=" * 40)

for name, array in EVALUATION_ARRAYS.items():
    print(
        f"{name}: "
        f"shape={array.shape}, "
        f"dtype={array.dtype}"
    )

EVALUATION DATA READY
X_threshold_seq: shape=(47408, 30, 42), dtype=float32
X_validation_seq: shape=(79031, 30, 42), dtype=float32
threshold_end_indices: shape=(47408,), dtype=int64
validation_end_indices: shape=(79031,), dtype=int64


In [13]:
# Prepare the existing evaluation arrays for subsequent model scoring.

X_threshold_seq = EVALUATION_ARRAYS["X_threshold_seq"]
X_validation_seq = EVALUATION_ARRAYS["X_validation_seq"]

threshold_end_indices = EVALUATION_ARRAYS["threshold_end_indices"]
validation_end_indices = EVALUATION_ARRAYS["validation_end_indices"]

# Validate sequence dimensions and index alignment.
assert X_threshold_seq.ndim == 3
assert X_validation_seq.ndim == 3

assert X_threshold_seq.shape[1:] == (30, 42)
assert X_validation_seq.shape[1:] == (30, 42)

assert len(X_threshold_seq) == len(threshold_end_indices)
assert len(X_validation_seq) == len(validation_end_indices)

print("Evaluation arrays prepared successfully.")
print(f"Threshold sequences: {X_threshold_seq.shape}")
print(f"Validation sequences: {X_validation_seq.shape}")
print(f"Threshold index range: {threshold_end_indices.min()}–{threshold_end_indices.max()}")
print(f"Validation index range: {validation_end_indices.min()}–{validation_end_indices.max()}")

Evaluation arrays prepared successfully.
Threshold sequences: (47408, 30, 42)
Validation sequences: (79031, 30, 42)
Threshold index range: 29–47436
Validation index range: 29–79059


In [14]:
import joblib
import numpy as np
import pandas as pd


import warnings
from sklearn.exceptions import InconsistentVersionWarning

warnings.filterwarnings(
    "ignore",
    category=InconsistentVersionWarning,
)

# Load the existing fitted models. Do not retrain them.
MODEL_DIR = PROJECT_ROOT / "swat-artifacts"

scaler = joblib.load(MODEL_DIR / "scaler.joblib")

isolation_forest = joblib.load(
    MODEL_DIR / "isolation_forest.joblib"
)

ocsvm = joblib.load(
    MODEL_DIR / "sgd_one_class_svm.joblib"
)

lof = joblib.load(
    MODEL_DIR / "lof_model.joblib"
)

# Use the feature order recorded by the fitted scaler.
FEATURE_COLUMNS = list(scaler.feature_names_in_)

if len(FEATURE_COLUMNS) != 42:
    raise ValueError(
        f"Expected 42 features, found {len(FEATURE_COLUMNS)}."
    )

print("Traditional ML artifacts loaded successfully.")
print(f"Feature count: {len(FEATURE_COLUMNS)}")

Traditional ML artifacts loaded successfully.
Feature count: 42


In [15]:
# Define the normal dataset path within the project.

DATA_RAW_DIR = PROJECT_ROOT / "swat-dataset"
NORMAL_PATH = DATA_RAW_DIR / "normal.csv"

if not NORMAL_PATH.is_file():
    raise FileNotFoundError(
        f"Normal dataset not found: {NORMAL_PATH}"
    )

print("Normal dataset path verified.")
# print(f"Path: {NORMAL_PATH}")

Normal dataset path verified.


In [16]:
# Reconstruct the established normal calibration and validation partitions.
import warnings

# Suppress pandas UserWarnings
warnings.filterwarnings('ignore', category=UserWarning)

TRAIN_ROWS = 221_366
MONITOR_ROWS = 47_435
THRESHOLD_ROWS = 47_437
VALIDATION_ROWS = 79_060

EXCLUDED_FEATURES = [
    "P202", "P204", "P206", "P301", "P401",
    "P404", "P502", "P601", "P603",
]

ALL_SENSOR_COLUMNS = FEATURE_COLUMNS + EXCLUDED_FEATURES

# Read the source data and reproduce Notebook 06's eligibility filter.
normal_df = pd.read_csv(NORMAL_PATH, low_memory=False)
normal_df.columns = normal_df.columns.str.strip()

missing_columns = [
    col for col in ["Timestamp", *ALL_SENSOR_COLUMNS]
    if col not in normal_df.columns
]

if missing_columns:
    raise KeyError(f"Required columns missing: {missing_columns}")

normal_df["Timestamp"] = pd.to_datetime(
    normal_df["Timestamp"],
    errors="coerce",
)

eligible_mask = (
    normal_df[ALL_SENSOR_COLUMNS].notna().all(axis=1)
    & normal_df["Timestamp"].notna()
)

eligible_normal = normal_df.loc[eligible_mask].copy()

# Verify the established partition sizes before extracting rows.
expected_rows = (
    TRAIN_ROWS + MONITOR_ROWS + THRESHOLD_ROWS + VALIDATION_ROWS
)

if len(eligible_normal) != expected_rows:
    raise ValueError(
        f"Expected {expected_rows:,} eligible observations, "
        f"but found {len(eligible_normal):,}."
    )

threshold_start = TRAIN_ROWS + MONITOR_ROWS
threshold_end = threshold_start + THRESHOLD_ROWS

threshold_df = eligible_normal.iloc[
    threshold_start:threshold_end
].copy()

validation_df = eligible_normal.iloc[threshold_end:].copy()

# Verify partition sizes and chronology.
assert len(threshold_df) == THRESHOLD_ROWS
assert len(validation_df) == VALIDATION_ROWS

assert threshold_df["Timestamp"].is_monotonic_increasing
assert validation_df["Timestamp"].is_monotonic_increasing

assert (
    threshold_df["Timestamp"].iloc[-1]
    < validation_df["Timestamp"].iloc[0]
)

print("NORMAL PARTITIONS RECONSTRUCTED")
print("=" * 48)
print(f"Eligible observations: {len(eligible_normal):,}")
print(f"Threshold observations: {len(threshold_df):,}")
print(f"Validation observations: {len(validation_df):,}")
print(
    "Threshold period:",
    threshold_df["Timestamp"].iloc[0],
    "to",
    threshold_df["Timestamp"].iloc[-1],
)
print(
    "Validation period:",
    validation_df["Timestamp"].iloc[0],
    "to",
    validation_df["Timestamp"].iloc[-1],
)
print("Chronological separation verified: True")

NORMAL PARTITIONS RECONSTRUCTED
Eligible observations: 395,298
Threshold observations: 47,437
Validation observations: 79,060
Threshold period: 2016-01-01 02:23:18 to 2016-01-01 15:51:19
Validation period: 2016-01-01 15:51:20 to 2016-01-02 14:59:59
Chronological separation verified: True


In [17]:
# Generate anomaly scores using the saved traditional ML models.
# Higher scores indicate greater anomalousness.

# Extract the established 42 features in the saved scaler's order.
X_threshold_raw = threshold_df[FEATURE_COLUMNS]
X_validation_raw = validation_df[FEATURE_COLUMNS]

# Apply the existing scaler; do not fit it again.
X_threshold = scaler.transform(X_threshold_raw).astype(np.float32)
X_validation = scaler.transform(X_validation_raw).astype(np.float32)

# Verify the expected dimensions and numerical validity.
assert X_threshold.shape == (THRESHOLD_ROWS, 42)
assert X_validation.shape == (VALIDATION_ROWS, 42)

assert np.isfinite(X_threshold).all()
assert np.isfinite(X_validation).all()

# 1. Isolation Forest
if_threshold_scores = -isolation_forest.score_samples(X_threshold)
if_validation_scores = -isolation_forest.score_samples(X_validation)

# 2. One-Class SVM
ocsvm_threshold_scores = -ocsvm.score_samples(X_threshold)
ocsvm_validation_scores = -ocsvm.score_samples(X_validation)

# 3. Local Outlier Factor (novelty mode)
lof_threshold_scores = -lof.score_samples(X_threshold)
lof_validation_scores = -lof.score_samples(X_validation)

# Verify score dimensions and numerical validity.
score_arrays = {
    "IF threshold": if_threshold_scores,
    "IF validation": if_validation_scores,
    "OCSVM threshold": ocsvm_threshold_scores,
    "OCSVM validation": ocsvm_validation_scores,
    "LOF threshold": lof_threshold_scores,
    "LOF validation": lof_validation_scores,
}

for name, scores in score_arrays.items():
    expected_length = (
        THRESHOLD_ROWS if "threshold" in name else VALIDATION_ROWS
    )

    if scores.shape != (expected_length,):
        raise ValueError(
            f"{name}: expected {(expected_length,)}, got {scores.shape}."
        )

    if not np.isfinite(scores).all():
        raise ValueError(f"{name} contains NaN or infinite values.")

print("TRADITIONAL ML SCORING COMPLETED")
print("=" * 60)

for name, scores in score_arrays.items():
    print(
        f"{name:20s} | "
        f"n={len(scores):,} | "
        f"min={scores.min():.6f} | "
        f"median={np.median(scores):.6f} | "
        f"max={scores.max():.6f}"
    )

print("\nAll six score arrays passed dimension and finiteness checks.")

TRADITIONAL ML SCORING COMPLETED
IF threshold         | n=47,437 | min=0.437794 | median=0.497420 | max=0.676735
IF validation        | n=79,060 | min=0.442309 | median=0.507095 | max=0.679314
OCSVM threshold      | n=47,437 | min=-15.859379 | median=-15.356825 | max=-13.448579
OCSVM validation     | n=79,060 | min=-16.083435 | median=-15.271841 | max=-9.913305
LOF threshold        | n=47,437 | min=1.008754 | median=10.446300 | max=59.167336
LOF validation       | n=79,060 | min=1.139901 | median=12.727806 | max=50.844120

All six score arrays passed dimension and finiteness checks.


In [18]:
# Persist traditional ML anomaly scores for Notebook 08 evaluation.

SCORES_FILE = MODEL_DIR / "traditional_ml_scores.npz"

np.savez_compressed(
    SCORES_FILE,
    if_threshold_scores=if_threshold_scores,
    if_validation_scores=if_validation_scores,
    ocsvm_threshold_scores=ocsvm_threshold_scores,
    ocsvm_validation_scores=ocsvm_validation_scores,
    lof_threshold_scores=lof_threshold_scores,
    lof_validation_scores=lof_validation_scores,
)

if not SCORES_FILE.is_file():
    raise FileNotFoundError(
        f"Traditional ML score archive was not created: {SCORES_FILE}"
    )

print("TRADITIONAL ML SCORES SAVED")
print("=" * 60)
# print(f"Path: {SCORES_FILE}")
print(f"File size: {SCORES_FILE.stat().st_size / (1024**2):.2f} MB")
print("Saved arrays:")
print("  - IF threshold")
print("  - IF validation")
print("  - OCSVM threshold")
print("  - OCSVM validation")
print("  - LOF threshold")
print("  - LOF validation")

TRADITIONAL ML SCORES SAVED
File size: 1.63 MB
Saved arrays:
  - IF threshold
  - IF validation
  - OCSVM threshold
  - OCSVM validation
  - LOF threshold
  - LOF validation


In [19]:
# Align traditional ML scores to the endpoint rows used by sequence models.

if_threshold_seq_scores = if_threshold_scores[threshold_end_indices]
if_validation_seq_scores = if_validation_scores[validation_end_indices]

ocsvm_threshold_seq_scores = ocsvm_threshold_scores[threshold_end_indices]
ocsvm_validation_seq_scores = ocsvm_validation_scores[validation_end_indices]

lof_threshold_seq_scores = lof_threshold_scores[threshold_end_indices]
lof_validation_seq_scores = lof_validation_scores[validation_end_indices]

# Verify alignment with the existing sequence arrays.
expected_threshold_sequences = len(X_threshold_seq)
expected_validation_sequences = len(X_validation_seq)

aligned_scores = {
    "IF threshold": if_threshold_seq_scores,
    "IF validation": if_validation_seq_scores,
    "OCSVM threshold": ocsvm_threshold_seq_scores,
    "OCSVM validation": ocsvm_validation_seq_scores,
    "LOF threshold": lof_threshold_seq_scores,
    "LOF validation": lof_validation_seq_scores,
}

for name, scores in aligned_scores.items():
    expected_length = (
        expected_threshold_sequences
        if "threshold" in name
        else expected_validation_sequences
    )

    if scores.shape != (expected_length,):
        raise ValueError(
            f"{name}: expected {(expected_length,)}, got {scores.shape}."
        )

    if not np.isfinite(scores).all():
        raise ValueError(f"{name} contains NaN or infinite values.")

print("TRADITIONAL ML SEQUENCE ALIGNMENT COMPLETED")
print("=" * 60)
print(f"Threshold sequence support:  {expected_threshold_sequences:,}")
print(f"Validation sequence support: {expected_validation_sequences:,}")
print()
print("All six aligned score arrays passed dimension and finiteness checks.")

TRADITIONAL ML SEQUENCE ALIGNMENT COMPLETED
Threshold sequence support:  47,408
Validation sequence support: 79,031

All six aligned score arrays passed dimension and finiteness checks.


In [20]:
# Score the saved LSTM Autoencoder on normal calibration and validation sequences.

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# 1. Reconstruct the exact architecture used in Notebook 05.

class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        bottleneck_size,
        dropout,
    ):
        super().__init__()

        self.encoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.encoder_projection = nn.Linear(
            hidden_size,
            bottleneck_size,
        )

        self.decoder_projection = nn.Linear(
            bottleneck_size,
            hidden_size,
        )

        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.output_layer = nn.Linear(
            hidden_size,
            input_size,
        )

    def forward(self, x):
        _, (hidden, _) = self.encoder(x)

        latent = F.leaky_relu(
            self.encoder_projection(hidden[-1]),
            negative_slope=0.01,
        )

        decoder_input = F.leaky_relu(
            self.decoder_projection(latent),
            negative_slope=0.01,
        )

        decoder_input = decoder_input.unsqueeze(1).repeat(
            1,
            x.size(1),
            1,
        )

        decoded, _ = self.decoder(decoder_input)

        return self.output_layer(decoded)


# 2. Select the available inference device.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. Load the saved model checkpoint.
LSTM_CHECKPOINT = MODEL_DIR / "lstm_autoencoder.pth"

if not LSTM_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"LSTM checkpoint not found: {LSTM_CHECKPOINT}"
    )

lstm_model = LSTMAutoencoder(
    input_size=42,
    hidden_size=64,
    num_layers=2,
    bottleneck_size=16,
    dropout=0.2,
).to(DEVICE)

try:
    lstm_state = torch.load(
        LSTM_CHECKPOINT,
        map_location=DEVICE,
        weights_only=True,
    )
except TypeError:
    lstm_state = torch.load(
        LSTM_CHECKPOINT,
        map_location=DEVICE,
    )

lstm_model.load_state_dict(lstm_state)
lstm_model.eval()


# 4. Calculate one MSE reconstruction score per sequence.
def calculate_reconstruction_errors(
    model,
    sequences,
    device,
    batch_size=512,
):
    model.eval()
    scores = []

    with torch.no_grad():
        for start in range(0, len(sequences), batch_size):
            batch = torch.as_tensor(
                np.asarray(
                    sequences[start:start + batch_size],
                    dtype=np.float32,
                ),
                dtype=torch.float32,
                device=device,
            )

            reconstruction = model(batch)

            batch_scores = (
                (reconstruction - batch) ** 2
            ).mean(dim=(1, 2))

            scores.append(batch_scores.cpu().numpy())

    return np.concatenate(scores).astype(np.float64)


# Confirm input dimensions before scoring.
assert X_threshold_seq.ndim == 3
assert X_validation_seq.ndim == 3
assert X_threshold_seq.shape[1:] == (30, 42)
assert X_validation_seq.shape[1:] == (30, 42)

lstm_threshold_scores = calculate_reconstruction_errors(
    model=lstm_model,
    sequences=X_threshold_seq,
    device=DEVICE,
)

lstm_validation_scores = calculate_reconstruction_errors(
    model=lstm_model,
    sequences=X_validation_seq,
    device=DEVICE,
)


# 5. Verify the outputs.
assert lstm_threshold_scores.shape == (len(X_threshold_seq),)
assert lstm_validation_scores.shape == (len(X_validation_seq),)
assert np.isfinite(lstm_threshold_scores).all()
assert np.isfinite(lstm_validation_scores).all()
assert (lstm_threshold_scores >= 0).all()
assert (lstm_validation_scores >= 0).all()


# 6. Save the LSTM scores for reproducible evaluation.
LSTM_SCORES_FILE = MODEL_DIR / "lstm_autoencoder_scores.npz"

np.savez_compressed(
    LSTM_SCORES_FILE,
    threshold_scores=lstm_threshold_scores,
    validation_scores=lstm_validation_scores,
    threshold_end_indices=threshold_end_indices,
    validation_end_indices=validation_end_indices,
)

print("LSTM AUTOENCODER SCORING COMPLETED")
print("=" * 60)
print(f"Device: {DEVICE}")
print(f"Threshold sequences:  {len(lstm_threshold_scores):,}")
print(f"Validation sequences: {len(lstm_validation_scores):,}")
print()
print("Threshold-calibration score distribution")
print(f"  Minimum: {lstm_threshold_scores.min():.8f}")
print(f"  Mean:    {lstm_threshold_scores.mean():.8f}")
print(f"  Median:  {np.median(lstm_threshold_scores):.8f}")
print(f"  95th pct: {np.percentile(lstm_threshold_scores, 95):.8f}")
print(f"  99th pct: {np.percentile(lstm_threshold_scores, 99):.8f}")
print(f"  Maximum: {lstm_threshold_scores.max():.8f}")
print()
print("Normal validation score distribution")
print(f"  Minimum: {lstm_validation_scores.min():.8f}")
print(f"  Mean:    {lstm_validation_scores.mean():.8f}")
print(f"  Median:  {np.median(lstm_validation_scores):.8f}")
print(f"  95th pct: {np.percentile(lstm_validation_scores, 95):.8f}")
print(f"  99th pct: {np.percentile(lstm_validation_scores, 99):.8f}")
print(f"  Maximum: {lstm_validation_scores.max():.8f}")
print()
# print(f"Scores saved to: {LSTM_SCORES_FILE}")
print("All score arrays passed validation.")

LSTM AUTOENCODER SCORING COMPLETED
Device: cuda
Threshold sequences:  47,408
Validation sequences: 79,031

Threshold-calibration score distribution
  Minimum: 0.00163359
  Mean:    0.00446212
  Median:  0.00324546
  95th pct: 0.01076058
  99th pct: 0.01882606
  Maximum: 0.15577848

Normal validation score distribution
  Minimum: 0.00311919
  Mean:    0.00686865
  Median:  0.00516798
  95th pct: 0.01582997
  99th pct: 0.02630818
  Maximum: 0.34991336

All score arrays passed validation.


In [21]:
# Score the saved TCN Autoencoder on normal calibration and validation sequences.
# 1. Reconstruct the exact TCN architecture from Notebook 07.

class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.left_padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0,
        )

    def forward(self, x):
        return self.conv(F.pad(x, (self.left_padding, 0)))


class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()

        self.conv1 = CausalConv1d(
            in_channels, out_channels, kernel_size, dilation
        )
        self.conv2 = CausalConv1d(
            out_channels, out_channels, kernel_size, dilation
        )

        self.activation = nn.ReLU()
        self.residual = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv1d(in_channels, out_channels, kernel_size=1)
        )

    def forward(self, x):
        residual = self.residual(x)
        x = self.activation(self.conv1(x))
        x = self.activation(self.conv2(x))
        return self.activation(x + residual)


class TCNAutoencoder(nn.Module):
    def __init__(self, n_features, channels, kernel_size, dilations):
        super().__init__()

        if len(channels) != len(dilations):
            raise ValueError("Each TCN block must have a dilation value.")

        encoder_layers = []
        in_channels = n_features

        for out_channels, dilation in zip(channels, dilations):
            encoder_layers.append(
                TCNBlock(
                    in_channels,
                    out_channels,
                    kernel_size,
                    dilation,
                )
            )
            in_channels = out_channels

        self.encoder = nn.Sequential(*encoder_layers)

        decoder_layers = []
        reversed_channels = list(reversed(channels))
        reversed_dilations = list(reversed(dilations))

        for index, dilation in enumerate(reversed_dilations):
            current_channels = reversed_channels[index]
            next_channels = (
                reversed_channels[index + 1]
                if index + 1 < len(reversed_channels)
                else n_features
            )

            decoder_layers.append(
                CausalConv1d(
                    current_channels,
                    next_channels,
                    kernel_size,
                    dilation,
                )
            )

            if index + 1 < len(reversed_dilations):
                decoder_layers.append(nn.ReLU())

        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError(
                "Expected input shape (batch, timesteps, features)."
            )

        x = x.transpose(1, 2).contiguous()
        x = self.encoder(x)
        x = self.decoder(x)

        return x.transpose(1, 2).contiguous()


# 2. Verify the expected sequence dimensions.
assert X_threshold_seq.shape[1:] == (30, 42)
assert X_validation_seq.shape[1:] == (30, 42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. Load the saved best TCN checkpoint.
TCN_CHECKPOINT = MODEL_DIR / "tcn_preliminary" / "tcn_autoencoder_best.pth"

if not TCN_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"TCN checkpoint not found: {TCN_CHECKPOINT}"
    )

tcn_model = TCNAutoencoder(
    n_features=42,
    channels=[32, 32, 32, 32],
    kernel_size=3,
    dilations=[1, 2, 4, 8],
).to(DEVICE)

try:
    tcn_state = torch.load(
        TCN_CHECKPOINT,
        map_location=DEVICE,
        weights_only=True,
    )
except TypeError:
    tcn_state = torch.load(
        TCN_CHECKPOINT,
        map_location=DEVICE,
    )

tcn_model.load_state_dict(tcn_state)
tcn_model.eval()


# 4. Calculate one mean squared reconstruction error per sequence.
def calculate_tcn_reconstruction_errors(
    model,
    sequences,
    device,
    batch_size=512,
):
    scores = []
    model.eval()

    with torch.no_grad():
        for start in range(0, len(sequences), batch_size):
            batch = torch.as_tensor(
                np.asarray(
                    sequences[start:start + batch_size],
                    dtype=np.float32,
                ),
                dtype=torch.float32,
                device=device,
            )

            reconstruction = model(batch)

            batch_scores = (
                (reconstruction - batch) ** 2
            ).mean(dim=(1, 2))

            scores.append(batch_scores.cpu().numpy())

    return np.concatenate(scores).astype(np.float64)


# 5. Score the normal calibration and validation sequences.
tcn_threshold_scores = calculate_tcn_reconstruction_errors(
    model=tcn_model,
    sequences=X_threshold_seq,
    device=DEVICE,
)

tcn_validation_scores = calculate_tcn_reconstruction_errors(
    model=tcn_model,
    sequences=X_validation_seq,
    device=DEVICE,
)


# 6. Validate the score arrays.
assert tcn_threshold_scores.shape == (len(X_threshold_seq),)
assert tcn_validation_scores.shape == (len(X_validation_seq),)
assert np.isfinite(tcn_threshold_scores).all()
assert np.isfinite(tcn_validation_scores).all()
assert (tcn_threshold_scores >= 0).all()
assert (tcn_validation_scores >= 0).all()


# 7. Save scores with their sequence endpoint indices.
TCN_SCORES_FILE = MODEL_DIR / "tcn_autoencoder_scores.npz"

np.savez_compressed(
    TCN_SCORES_FILE,
    threshold_scores=tcn_threshold_scores,
    validation_scores=tcn_validation_scores,
    threshold_end_indices=threshold_end_indices,
    validation_end_indices=validation_end_indices,
)

# 8. Report the observed score distributions.
print("TCN AUTOENCODER SCORING COMPLETED")
print("=" * 60)
print(f"Device: {DEVICE}")
print(f"Threshold sequences:  {len(tcn_threshold_scores):,}")
print(f"Validation sequences: {len(tcn_validation_scores):,}")
print()

for name, scores in [
    ("Threshold calibration", tcn_threshold_scores),
    ("Normal validation", tcn_validation_scores),
]:
    print(name)
    print(f"  Minimum:  {scores.min():.8f}")
    print(f"  Mean:     {scores.mean():.8f}")
    print(f"  Median:   {np.median(scores):.8f}")
    print(f"  95th pct: {np.percentile(scores, 95):.8f}")
    print(f"  99th pct: {np.percentile(scores, 99):.8f}")
    print(f"  Maximum:  {scores.max():.8f}")
    print()

# print(f"Scores saved to: {TCN_SCORES_FILE}")
print("All score arrays passed validation.")

TCN AUTOENCODER SCORING COMPLETED
Device: cuda
Threshold sequences:  47,408
Validation sequences: 79,031

Threshold calibration
  Minimum:  0.00037321
  Mean:     0.00131288
  Median:   0.00109156
  95th pct: 0.00253684
  99th pct: 0.00390673
  Maximum:  0.00703319

Normal validation
  Minimum:  0.00063398
  Mean:     0.00207530
  Median:   0.00158158
  95th pct: 0.00338929
  99th pct: 0.00588506
  Maximum:  0.14281602

All score arrays passed validation.


In [22]:
# Inspect the attack dataset schema and a small sample.

import pandas as pd

ATTACK_PATH = DATA_RAW_DIR / "attack.csv"

if not ATTACK_PATH.is_file():
    raise FileNotFoundError(f"Attack dataset not found: {ATTACK_PATH}")

attack_sample = pd.read_csv(
    ATTACK_PATH,
    nrows=10,
    low_memory=False
)

print("ATTACK DATASET INSPECTION")
print("=" * 60)
# print(f"File: {ATTACK_PATH}")
print(f"Columns: {len(attack_sample.columns)}")
print("\nColumn names:")
print(attack_sample.columns.tolist())

print("\nData types:")
print(attack_sample.dtypes.to_string())

print("\nFirst five rows:")
display(attack_sample.head())

print("\nMissing values in the sample:")
print(attack_sample.isna().sum().to_string())

ATTACK DATASET INSPECTION
Columns: 53

Column names:
[' Timestamp', 'FIT101', 'LIT101', ' MV101', 'P101', 'P102', ' AIT201', 'AIT202', 'AIT203', 'FIT201', ' MV201', ' P201', ' P202', 'P203', ' P204', 'P205', 'P206', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', ' MV303', 'MV304', 'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P401', 'P402', 'P403', 'P404', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'P502', 'PIT501', 'PIT502', 'PIT503', 'FIT601', 'P601', 'P602', 'P603', 'Normal/Attack']

Data types:
 Timestamp        object
FIT101           float64
LIT101           float64
 MV101             int64
P101               int64
P102               int64
 AIT201          float64
AIT202           float64
AIT203           float64
FIT201             int64
 MV201             int64
 P201              int64
 P202              int64
P203               int64
 P204              int64
P205               int64
P206               int64
DPIT

,Timestamp,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,AIT203,FIT201,...,P501,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603,Normal/Attack
0,28/12/2015 10:29:14 AM,2.493675,817.6742,2,1,1,262.625,8.46533,319.7385,0,...,2,1,249.8718,1.681991,188.7977,0.000128,1,1,1,Attack
1,28/12/2015 10:29:15 AM,2.535951,817.9490,2,1,1,262.625,8.46533,319.7385,0,...,2,1,249.8718,1.681991,188.7977,0.000128,1,1,1,Attack
2,28/12/2015 10:29:16 AM,2.535951,817.9490,2,1,1,262.625,8.46533,319.7385,0,...,2,1,249.8718,1.681991,188.7977,0.000128,1,1,1,Attack
3,28/12/2015 10:29:17 AM,2.569900,818.4592,2,1,1,262.625,8.46533,319.7385,0,...,2,1,249.8718,1.681991,188.7977,0.000128,1,1,1,Attack
4,28/12/2015 10:29:18 AM,2.610575,818.8911,2,1,1,262.625,8.46533,319.7385,0,...,2,1,250.0801,1.681991,188.9740,0.000128,1,1,1,Attack



Missing values in the sample:
 Timestamp       0
FIT101           0
LIT101           0
 MV101           0
P101             0
P102             0
 AIT201          0
AIT202           0
AIT203           0
FIT201           0
 MV201           0
 P201            0
 P202            0
P203             0
 P204            0
P205             0
P206             0
DPIT301          0
FIT301           0
LIT301           0
MV301            0
MV302            0
 MV303           0
MV304            0
P301             0
P302             0
AIT401           0
AIT402           0
FIT401           0
LIT401           0
P401             0
P402             0
P403             0
P404             0
UV401            0
AIT501           0
AIT502           0
AIT503           0
AIT504           0
FIT501           0
FIT502           0
FIT503           0
FIT504           0
P501             0
P502             0
PIT501           0
PIT502           0
PIT503           0
FIT601           0
P601             0
P602             0


In [23]:
# Validate the attack dataset in chunks before model evaluation.
import csv

CHUNK_SIZE = 50_000

total_rows = 0
invalid_timestamps = 0
missing_timestamps = 0
out_of_order_chunks = 0
missing_values_by_column = None
label_counts = {}
first_timestamp = None
last_timestamp = None
previous_timestamp = None
timestamp_format = "%d/%m/%Y %I:%M:%S %p"

with ATTACK_PATH.open("r", encoding="utf-8-sig", newline="") as file:
    reader = csv.reader(file)
    raw_header = next(reader)

normalized_header = [column.strip() for column in raw_header]

if len(normalized_header) != len(set(normalized_header)):
    raise ValueError("Duplicate column names exist after whitespace normalization.")

required_columns = {"Timestamp", "Normal/Attack", *FEATURE_COLUMNS}
missing_required = required_columns - set(normalized_header)

if missing_required:
    raise ValueError(
        f"Required columns missing from attack.csv: {sorted(missing_required)}"
    )

for chunk in pd.read_csv(
    ATTACK_PATH,
    chunksize=CHUNK_SIZE,
    low_memory=False,
    encoding="utf-8-sig",
):
    chunk.columns = chunk.columns.str.strip()
    chunk_rows = len(chunk)
    total_rows += chunk_rows

    if missing_values_by_column is None:
        missing_values_by_column = pd.Series(0, index=chunk.columns, dtype="int64")

    missing_values_by_column = (
        missing_values_by_column.add(chunk.isna().sum(), fill_value=0)
        .astype("int64")
    )

    labels = chunk["Normal/Attack"].astype("string").str.strip()
    counts = labels.value_counts(dropna=False)

    for label, count in counts.items():
        label_counts[str(label)] = label_counts.get(str(label), 0) + int(count)

    timestamps = pd.to_datetime(
        chunk["Timestamp"].astype("string").str.strip(),
        format=timestamp_format,
        errors="coerce",
    )

    invalid_timestamps += int(timestamps.isna().sum())
    valid_timestamps = timestamps.dropna()

    if len(valid_timestamps) != len(timestamps):
        missing_timestamps += int(timestamps.isna().sum())

    if not valid_timestamps.empty:
        chunk_first = valid_timestamps.iloc[0]
        chunk_last = valid_timestamps.iloc[-1]

        if first_timestamp is None:
            first_timestamp = chunk_first

        if previous_timestamp is not None and chunk_first < previous_timestamp:
            out_of_order_chunks += 1

        if not timestamps.is_monotonic_increasing:
            out_of_order_chunks += 1

        previous_timestamp = chunk_last
        last_timestamp = chunk_last

print("ATTACK DATASET VALIDATION")
print("=" * 60)
print(f"Total rows: {total_rows:,}")
print(f"Total columns: {len(normalized_header)}")
print(f"Invalid timestamps: {invalid_timestamps:,}")
print(f"Missing timestamps: {missing_timestamps:,}")
print(f"Out-of-order checks detected: {out_of_order_chunks}")
print(f"First valid timestamp: {first_timestamp}")
print(f"Last valid timestamp: {last_timestamp}")

print("\nLabel distribution:")
for label, count in sorted(label_counts.items()):
    print(f"  {label}: {count:,}")

print("\nColumns with missing values:")
if missing_values_by_column is None:
    print("No data rows were read.")
else:
    missing_columns = missing_values_by_column[
        missing_values_by_column > 0
    ].sort_values(ascending=False)

    if missing_columns.empty:
        print("None.")
    else:
        print(missing_columns.to_string())

print("\nFeature compatibility:")
print(f"Expected model features: {len(FEATURE_COLUMNS)}")
print(f"All expected features present: {not missing_required}")

ATTACK DATASET VALIDATION
Total rows: 54,621
Total columns: 53
Invalid timestamps: 0
Missing timestamps: 0
Out-of-order checks detected: 0
First valid timestamp: 2015-12-28 10:29:14
Last valid timestamp: 2016-01-02 13:41:11

Label distribution:
  Attack: 54,621

Columns with missing values:
None.

Feature compatibility:
Expected model features: 42
All expected features present: True


In [24]:
# Prepare attack evaluation sequences.
# Reuse the existing scaler; do not retrain or refit anything.

WINDOW_SIZE = 30

# Load the attack dataset and normalize column names.
attack_df = pd.read_csv(ATTACK_PATH, low_memory=False)
attack_df.columns = attack_df.columns.str.strip()

# Parse timestamps using the verified format.
attack_df["Timestamp"] = pd.to_datetime(
    attack_df["Timestamp"].astype("string").str.strip(),
    format="%d/%m/%Y %I:%M:%S %p",
    errors="raise",
)

# Preserve and validate the ground-truth labels.
attack_labels = attack_df["Normal/Attack"].astype("string").str.strip()

assert attack_labels.eq("Attack").all(), (
    "Unexpected labels found in the attack dataset."
)
assert attack_df["Timestamp"].is_monotonic_increasing, (
    "Attack observations are not in chronological order."
)
assert attack_df["Timestamp"].is_unique, (
    "Duplicate attack timestamps were found."
)
assert all(column in attack_df.columns for column in FEATURE_COLUMNS), (
    "One or more required model features are missing."
)

# Transform using the existing scaler fitted on training data.
X_attack_scaled = scaler.transform(
    attack_df[FEATURE_COLUMNS]
).astype(np.float32)

# Construct independent 30-observation windows.
n_sequences = len(X_attack_scaled) - WINDOW_SIZE + 1

if n_sequences <= 0:
    raise ValueError("Not enough observations to construct attack sequences.")

attack_sequences = np.stack([
    X_attack_scaled[start:start + WINDOW_SIZE]
    for start in range(n_sequences)
]).astype(np.float32)

# Each sequence is associated with its final observation.
attack_end_indices = np.arange(
    WINDOW_SIZE - 1,
    len(attack_df),
    dtype=np.int64,
)

attack_sequence_timestamps = (
    attack_df["Timestamp"]
    .iloc[attack_end_indices]
    .reset_index(drop=True)
)

attack_sequence_labels = (
    attack_labels
    .iloc[attack_end_indices]
    .reset_index(drop=True)
)

# Validate sequence dimensions and label alignment.
assert attack_sequences.shape == (
    54_592,
    WINDOW_SIZE,
    len(FEATURE_COLUMNS),
)
assert len(attack_end_indices) == len(attack_sequences)
assert len(attack_sequence_timestamps) == len(attack_sequences)
assert len(attack_sequence_labels) == len(attack_sequences)
assert np.isfinite(attack_sequences).all()
assert attack_sequence_labels.eq("Attack").all()

# Save the prepared sequences and endpoint indices.
ATTACK_SEQUENCES_FILE = MODEL_DIR / "attack_evaluation_sequences.npz"

np.savez_compressed(
    ATTACK_SEQUENCES_FILE,
    X_attack_seq=attack_sequences,
    attack_end_indices=attack_end_indices,
)

print("ATTACK SEQUENCE PREPARATION COMPLETED")
print("=" * 60)
print(f"Attack observations: {len(attack_df):,}")
print(f"Window size: {WINDOW_SIZE}")
print(f"Attack sequences: {len(attack_sequences):,}")
print(f"Sequence shape: {attack_sequences.shape}")
print(f"First sequence endpoint: {attack_sequence_timestamps.iloc[0]}")
print(f"Last sequence endpoint: {attack_sequence_timestamps.iloc[-1]}")
print(
    "All sequence labels are Attack: "
    f"{attack_sequence_labels.eq('Attack').all()}"
)
print(
    "All scaled values are finite: "
    f"{np.isfinite(attack_sequences).all()}"
)
# print(f"Saved to: {ATTACK_SEQUENCES_FILE}")

ATTACK SEQUENCE PREPARATION COMPLETED
Attack observations: 54,621
Window size: 30
Attack sequences: 54,592
Sequence shape: (54592, 30, 42)
First sequence endpoint: 2015-12-28 10:29:43
Last sequence endpoint: 2016-01-02 13:41:11
All sequence labels are Attack: True
All scaled values are finite: True


In [25]:
# Inspect the saved traditional ML model metadata.

import json

METADATA_PATH = MODEL_DIR / "traditional_ml_model_metadata.json"

if not METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"Traditional ML metadata not found: {METADATA_PATH}"
    )

with METADATA_PATH.open("r", encoding="utf-8") as file:
    traditional_metadata = json.load(file)

print("SAVED TRADITIONAL ML MODEL METADATA")
print("=" * 60)
print(json.dumps(traditional_metadata, indent=2))

SAVED TRADITIONAL ML MODEL METADATA
{
  "notebook": "06_traditional_ml_anomaly_detection",
  "scikit_learn_version": "1.6.1",
  "feature_count": 42,
  "feature_columns": [
    "FIT101",
    "LIT101",
    "MV101",
    "P101",
    "P102",
    "AIT201",
    "AIT202",
    "AIT203",
    "FIT201",
    "MV201",
    "P201",
    "P203",
    "P205",
    "DPIT301",
    "FIT301",
    "LIT301",
    "MV301",
    "MV302",
    "MV303",
    "MV304",
    "P302",
    "AIT401",
    "AIT402",
    "FIT401",
    "LIT401",
    "P402",
    "P403",
    "UV401",
    "AIT501",
    "AIT502",
    "AIT503",
    "AIT504",
    "FIT501",
    "FIT502",
    "FIT503",
    "FIT504",
    "P501",
    "PIT501",
    "PIT502",
    "PIT503",
    "FIT601",
    "P602"
  ],
  "training_observations": 221366,
  "training_features": 42,
  "models": {
    "isolation_forest": {
      "artifact": "isolation_forest.joblib",
      "n_estimators": 200,
      "max_samples": "auto",
      "contamination": "auto",
      "random_state": 42
   

In [26]:
# Inspect the existing LOF configuration and saved model metadata.

import json

METADATA_PATH = MODEL_DIR / "traditional_ml_model_metadata.json"

with METADATA_PATH.open("r", encoding="utf-8") as file:
    traditional_metadata = json.load(file)

print("RECORDED TRADITIONAL ML CONFIGURATIONS")
print("=" * 60)

for model_name, configuration in traditional_metadata.get("models", {}).items():
    print(f"\n{model_name}")
    print("-" * 40)
    print(json.dumps(configuration, indent=2))

RECORDED TRADITIONAL ML CONFIGURATIONS

isolation_forest
----------------------------------------
{
  "artifact": "isolation_forest.joblib",
  "n_estimators": 200,
  "max_samples": "auto",
  "contamination": "auto",
  "random_state": 42
}

sgd_one_class_svm
----------------------------------------
{
  "artifact": "sgd_one_class_svm.joblib",
  "implementation": "SGDOneClassSVM",
  "kernel": "linear",
  "nu": 0.01,
  "max_iter": 80,
  "tol": null,
  "random_state": 42
}


In [27]:
# Inspect the configuration of the already-loaded LOF model.

if "lof" not in globals():
    raise NameError(
        "The LOF model is not loaded. Load the existing saved artifact "
        "before running this inspection."
    )

print("SAVED LOF MODEL CONFIGURATION")
print("=" * 60)

for parameter in [
    "n_neighbors",
    "novelty",
    "metric",
    "p",
    "contamination",
    "algorithm",
    "leaf_size",
    "n_jobs",
]:
    if hasattr(lof, parameter):
        print(f"{parameter}: {getattr(lof, parameter)}")

print("\nTraining observations:", getattr(lof, "n_samples_fit_", "Not available"))
print("Training feature count:", getattr(lof, "n_features_in_", "Not available"))

SAVED LOF MODEL CONFIGURATION
n_neighbors: 20
novelty: True
metric: minkowski
p: 2
contamination: auto
algorithm: auto
leaf_size: 30
n_jobs: -1

Training observations: 221366
Training feature count: 42


In [28]:
# Score the existing USAD checkpoint.
# No retraining, hyperparameter tuning, or scaler refitting.


USAD_CHECKPOINT = MODEL_DIR / "usad_preliminary" / "usad_best.pth"

if not USAD_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"USAD checkpoint not found: {USAD_CHECKPOINT}"
    )

# Reconstruct the exact saved architecture.
class USAD(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
        )

        self.decoder1 = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

        self.decoder2 = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        latent = self.encoder(x)
        w1 = self.decoder1(latent)
        w2 = self.decoder2(latent)
        w3 = self.decoder2(self.encoder(w1))
        return w1, w2, w3


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    checkpoint = torch.load(
        USAD_CHECKPOINT,
        map_location=DEVICE,
        weights_only=True,
    )
except TypeError:
    checkpoint = torch.load(
        USAD_CHECKPOINT,
        map_location=DEVICE,
    )

if not isinstance(checkpoint, dict) or "model_state_dict" not in checkpoint:
    raise ValueError(
        "The USAD checkpoint does not have the expected saved structure."
    )

usad_config = checkpoint["config"]

assert usad_config["sequence_length"] == 30
assert usad_config["feature_count"] == 42
assert usad_config["input_dim"] == 1260

usad_model = USAD(
    input_dim=usad_config["input_dim"],
    hidden_dim=usad_config["hidden_dim"],
    latent_dim=usad_config["latent_dim"],
).to(DEVICE)

usad_model.load_state_dict(checkpoint["model_state_dict"])
usad_model.eval()

def score_usad_sequences(model, sequences, device, batch_size=512):
    """Return one USAD reconstruction score per sequence."""
    scores = []

    model.eval()

    with torch.no_grad():
        for start in range(0, len(sequences), batch_size):
            batch_sequences = np.asarray(
                sequences[start:start + batch_size],
                dtype=np.float32,
            )

            batch = torch.as_tensor(
                batch_sequences.reshape(len(batch_sequences), -1),
                dtype=torch.float32,
                device=device,
            )

            w1, w2, _ = model(batch)

            mse_w1 = ((w1 - batch) ** 2).mean(dim=1)
            mse_w2 = ((w2 - batch) ** 2).mean(dim=1)

            batch_scores = 0.5 * (mse_w1 + mse_w2)
            scores.append(batch_scores.cpu().numpy())

    return np.concatenate(scores).astype(np.float64)


usad_threshold_scores = score_usad_sequences(
    usad_model,
    X_threshold_seq,
    DEVICE,
)

usad_validation_scores = score_usad_sequences(
    usad_model,
    X_validation_seq,
    DEVICE,
)

assert usad_threshold_scores.shape == (len(X_threshold_seq),)
assert usad_validation_scores.shape == (len(X_validation_seq),)
assert np.isfinite(usad_threshold_scores).all()
assert np.isfinite(usad_validation_scores).all()
assert (usad_threshold_scores >= 0).all()
assert (usad_validation_scores >= 0).all()

USAD_SCORES_FILE = MODEL_DIR / "usad_preliminary_scores.npz"

np.savez_compressed(
    USAD_SCORES_FILE,
    threshold_scores=usad_threshold_scores,
    validation_scores=usad_validation_scores,
    threshold_end_indices=threshold_end_indices,
    validation_end_indices=validation_end_indices,
)

print("USAD CHECKPOINT SCORING COMPLETED")
print("=" * 60)
print(f"Device: {DEVICE}")
print(f"Checkpoint best epoch: {checkpoint['best_epoch']}")
print(
    "Best monitoring reconstruction MSE: "
    f"{checkpoint['best_monitor_reconstruction_mse']:.8f}"
)
print(f"Threshold sequences: {len(usad_threshold_scores):,}")
print(f"Validation sequences: {len(usad_validation_scores):,}")

for name, scores in [
    ("Threshold calibration", usad_threshold_scores),
    ("Normal validation", usad_validation_scores),
]:
    print(f"\n{name}")
    print(f"  Minimum: {scores.min():.8f}")
    print(f"  Mean:    {scores.mean():.8f}")
    print(f"  Median:  {np.median(scores):.8f}")
    print(f"  95th pct: {np.percentile(scores, 95):.8f}")
    print(f"  99th pct: {np.percentile(scores, 99):.8f}")
    print(f"  Maximum: {scores.max():.8f}")

# print(f"\nScores saved to: {USAD_SCORES_FILE}")
print("All score arrays passed validation.")

USAD CHECKPOINT SCORING COMPLETED
Device: cuda
Checkpoint best epoch: 1
Best monitoring reconstruction MSE: 0.02222458
Threshold sequences: 47,408
Validation sequences: 79,031

Threshold calibration
  Minimum: 0.01278505
  Mean:    0.02537261
  Median:  0.02130574
  95th pct: 0.04379082
  99th pct: 0.05641948
  Maximum: 0.07907878

Normal validation
  Minimum: 0.01818857
  Mean:    0.02952056
  Median:  0.02564091
  95th pct: 0.04864919
  99th pct: 0.06123613
  Maximum: 0.13038802
All score arrays passed validation.


# 08 — Threshold Calibration and Evaluation

## 1. Purpose

This notebook establishes a controlled evaluation procedure for the nine anomaly-detection methods developed in the preceding notebooks.

The objectives are to:

- preserve the existing model configurations and baseline definitions during the initial audit;
- establish a consistent procedure for generating and storing anomaly scores;
- calibrate learned-model decision thresholds using the designated normal threshold-calibration partition;
- evaluate the methods using the normal validation and attack datasets;
- document differences in feature sets, scoring granularity, and threshold selection;
- report detection performance without overstating temporal independence or generalizability.

The evaluation must distinguish between model training, model-configuration tuning, threshold calibration, and final evaluation. These are separate activities and must not be treated as interchangeable.

## 2. Evaluation Data and Temporal Limitations

### 2.1 Normal Data Partitions

The eligible normal dataset contains 395,298 observations after the exclusions documented in Notebook 02.

The chronological partitioning established in the preceding notebooks is:

| Partition             | Observations | Timestamp range                            |
| --------------------- | -----------: | ------------------------------------------ |
| Training              |      221,366 | 2015-12-28 10:00:00 to 2015-12-31 12:43:42 |
| Monitoring            |       47,435 | 2015-12-31 12:43:43 to 2016-01-01 02:23:17 |
| Threshold calibration |       47,437 | 2016-01-01 02:23:18 to 2016-01-01 15:51:19 |
| Normal validation     |       79,060 | 2016-01-01 15:51:20 to 2016-01-02 14:59:59 |
| **Total**             |  **395,298** |                                            |

The training, monitoring, and threshold-calibration partitions together form the 316,238-observation calibration period used by the baseline notebook.

The normal validation partition is the same 79,060-observation partition used by both the baseline and learned-model workflows.

For sequence-based models, the existing 30-observation windows exclude the first 29 observations of each independently prepared partition from endpoint scoring. Consequently, the number of sequence scores is smaller than the number of source observations.

The established partition boundaries and sequence endpoint indices must be preserved.

### 2.2 Attack Data

The attack dataset contains 54,621 observations, all labelled `Attack`.

The attack dataset was reserved for evaluation and was not used for model fitting, hyperparameter selection, or threshold calibration. Its recorded timestamp range overlaps the timestamp range of the normal-validation partition, although the existing comparison found no exact timestamps shared between the two datasets. The attack dataset is therefore held out by data-use protocol, but the combined evaluation is not temporally disjoint by timestamp range

Notebook 02, Section 17, records the following timestamp-overlap findings:

| Check                                                        | Recorded result |
| ------------------------------------------------------------ | --------------: |
| Unique normal timestamps                                     |         892,098 |
| Unique attack timestamps                                     |          54,621 |
| Shared exact timestamps                                      |             0   |
| Unique normal timestamps within the attack file's time range |         388,816 |
| Attack timestamps within the normal file's time range        |          54,621 |

The normal and attack files share no exact timestamps. However, their overall wall-clock ranges overlap. All attack timestamps fall within the overall time range of the normal file.

These counts describe the timestamp sets examined in Notebook 02. They do not establish the exact number of observations in the final normal validation partition that fall within the attack interval.

**Evaluation limitation:** Evaluation using `attack.csv` and normal validation must be treated as a *temporally overlapping evaluation, not as a temporally independent hold-out*.

The combined evaluation can support descriptive comparisons of detection performance on the available data. It cannot, by itself, establish performance on a temporally independent future period.

This limitation must accompany the reported evaluation results.

### 2.3 Evaluation Classes

For the combined evaluation:

- The normal validation partition provides the negative-class observations.
- The attack dataset provides the positive-class observations.
- Attack labels are reserved for evaluation and must not be used to train models, select model configurations, or calibrate decision thresholds.

Because `attack.csv` contains no normal-labelled observations, it cannot independently provide both classes required for a binary confusion matrix.

The use of normal validation and attack observations to calculate confusion-matrix metrics must be accompanied by the temporal-overlap limitation described above.

## 3. The Nine Anomaly-Detection Methods

The evaluation covers the following nine methods.

| Category                     | Method                                   |
| ---------------------------- | ---------------------------------------- |
| Baseline                     | Rule-Based Detection                     |
| Baseline                     | Rolling Statistical Detection            |
| Baseline                     | Simple Statistical Threshold             |
| Traditional machine learning | Isolation Forest                         |
| Traditional machine learning | SGD One-Class SVM                        |
| Traditional machine learning | Local Outlier Factor (LOF), novelty mode |
| Deep learning                | LSTM Autoencoder                         |
| Deep learning                | TCN Autoencoder                          |
| Deep learning                | USAD                                     |

The three baselines and six learned models remain part of the project.

The initial evaluation stage uses the existing baseline definitions and saved learned-model checkpoints. Any subsequent model-configuration changes must be justified by the configuration audit and documented separately.

## 4. Feature Sets and Input Transformations

### 4.1 Baseline Methods

The three baseline methods use all 51 sensor columns identified in Notebook 04.

They operate on raw sensor values without the learned models' MinMax scaling.

Their scores and decisions must retain the definitions established in Notebook 04.

### 4.2 Learned Models

Isolation Forest, SGD One-Class SVM, LOF, LSTM Autoencoder, TCN Autoencoder, and USAD use the established 42-feature set.

The feature order must match the fitted scaler's `feature_names_in_` and the order used by the saved model artifacts.

The existing `MinMaxScaler` was fitted on the designated normal training partition. It must be reused for subsequent inference; it must not be refitted on the threshold-calibration, validation, or attack data.

Scaled observations may fall slightly outside the nominal training range when new observations extend beyond the fitted minimum or maximum. This behaviour must not be corrected by refitting the scaler on evaluation data.

### 4.3 Implications for Comparisons

The baseline methods and learned models do not use identical feature sets or input transformations.

Consequently, comparisons across all nine methods must acknowledge that:

- the baselines use 51 raw sensor features;
- the learned models use 42 selected sensor features;
- the learned models use the existing training-fitted scaler;
- the methods use different scoring functions and may operate at different temporal granularities.

A difference in detection performance cannot be attributed solely to the algorithm when the input feature sets and transformations also differ.

## 5. Baseline Definitions and Frozen Decision Rules

The baseline definitions are established in Notebook 04. They must not be silently recalibrated during this notebook.

### 5.1 Rule-Based Detection

For each sensor, the lower and upper reference limits are the 1st and 99th percentiles of that sensor's values in the normal calibration partition.

An observation's anomaly score is the number of sensors whose values fall outside their corresponding empirical limits.

The existing decision rule flags an observation when at least one sensor violates its limits:

$$
S_t = \sum_{j=1}^{51}
\mathbf{1}\left(x_{t,j}<L_j \;\lor\; x_{t,j}>U_j\right)
$$

$$
\hat{y}_t = \mathbf{1}(S_t>0)
$$

Here, \(L_j\) and \(U_j\) denote the empirical lower and upper reference limits for sensor \(j\).

These are data-derived reference limits, not validated industrial safety limits or documented plant operating constraints.

### 5.2 Rolling Statistical Detection

The existing rolling detector uses:

- a 60-observation causal rolling window;
- the preceding observations only, excluding the current observation;
- the sample standard deviation, with `ddof=1`;
- an absolute rolling z-score threshold of 3.0.

For a sensor with a valid rolling standard deviation:

$$
z_{t,j} =
\frac{x_{t,j}-\mu_{t-1,j}^{(60)}}
{\sigma_{t-1,j}^{(60)}}
$$

The observation-level anomaly score is the number of sensors whose absolute rolling z-scores exceed 3.0:

$$
S_t = \sum_j \mathbf{1}(|z_{t,j}|>3.0)
$$

An observation is flagged when at least one scorable sensor exceeds the threshold.

Zero rolling standard deviations are treated as undefined, as in the existing implementation. The resulting unscorable values and their implications must be preserved and documented.

The rolling detector's history must follow the established chronological processing procedure.

### 5.3 Simple Statistical Threshold

For each sensor, the existing detector uses the mean and sample standard deviation calculated from the normal calibration partition.

The absolute standardized deviation is:

$$
z_{t,j} = \frac{x_{t,j}-\mu_j}{\sigma_j}
$$

The observation-level anomaly score counts the scorable sensors for which:

$$
|z_{t,j}|>3.0
$$

An observation is flagged when at least one scorable sensor exceeds the threshold.

The existing implementation masks zero-standard-deviation sensors. It must not assign an artificial nonzero variance to those sensors merely to make them scorable.

### 5.4 Baseline Threshold Policy

The following decisions are fixed for this evaluation:

- The rule-based detector retains its empirical 1st/99th-percentile limits and zero-violation decision boundary.
- The rolling detector retains its 60-observation window and absolute z-score threshold of 3.0.
- The simple statistical detector retains its calibration-derived means and standard deviations and absolute z-score threshold of 3.0.
- The three baselines will not be recalibrated using the learned models' threshold-calibration partition.
- The baseline definitions will not be changed to improve performance against attack labels.

This preserves the distinction between the established baselines and the learned models' threshold-calibration procedure.

## 6. Learned-Model Configuration Audit

Before any tuning, the six learned models must undergo a configuration audit.

The audit must establish, using the existing notebooks, metadata, saved checkpoints, and scoring code:

1. The configuration actually used for each model.
2. Whether the saved checkpoint can be loaded using the expected architecture.
3. Whether the existing score-generation procedure matches the model's documented scoring definition.
4. Whether the required model and data artifacts are present.
5. Whether the existing configuration and training record support a defensible evaluation.
6. Whether controlled model-configuration tuning is justified.

The audit must distinguish confirmed facts from unresolved questions.

No model must be retrained or tuned merely because tuning is possible. Any approved tuning experiment must have a documented rationale, a bounded search space, separate output paths, and a clear separation between development data and final evaluation data.

The three baselines are not part of this learned-model tuning stage.

## 7. Threshold Calibration for Learned Models

The 47,437-observation normal threshold-calibration partition is reserved for establishing anomaly-detection thresholds for the learned models.

The threshold-calibration partition must remain separate from:

- the model-training partition;
- the monitoring partition used during model development;
- the final normal validation partition;
- the attack evaluation data.

Model configurations must be fixed before their final thresholds are established.

For each learned model, the threshold-calibration procedure must:

1. Generate anomaly scores using the existing fitted model and preprocessing pipeline.
2. Confirm score-array dimensions, finiteness, and endpoint alignment.
3. Apply a documented threshold-selection rule using normal threshold-calibration scores only.
4. Record the selected threshold and its rationale.
5. Freeze the threshold before evaluating the normal validation and attack datasets.

No threshold may be selected by maximizing attack-label performance.

The exact threshold-selection rule and any parameters it requires must be documented before threshold calibration is performed. The rule must not be changed after examining final evaluation results to improve reported performance.

## 8. Anomaly Scores and Temporal Alignment

Anomaly scores are not directly interchangeable across methods.

The existing score definitions are:

| Method or family              | Score definition                                                          |
| ----------------------------- | ------------------------------------------------------------------------- |
| Rule-Based Detection          | Number of sensor-limit violations                                         |
| Rolling Statistical Detection | Number of sensors exceeding the rolling z-score threshold                 |
| Simple Statistical Threshold  | Number of sensors exceeding the fixed z-score threshold                   |
| Isolation Forest              | Negative `score_samples` value                                            |
| SGD One-Class SVM             | Negative `score_samples` value                                            |
| LOF                           | Negative `score_samples` value in novelty mode                            |
| LSTM Autoencoder              | Mean squared reconstruction error across time steps and features          |
| TCN Autoencoder               | Mean squared reconstruction error across time steps and features          |
| USAD                          | Mean of the reconstruction MSEs from its first and second decoder outputs |

For the learned models, larger scores are interpreted as greater anomaly evidence under the established score convention.

The baseline scores are integer-valued counts, whereas the learned-model scores have method-specific numerical scales. Their raw score magnitudes must not be compared as if they represented a common unit.

For sequence-based models, each score corresponds to the endpoint of a 30-observation input window. Scores must be aligned using the established endpoint indices rather than assigned indiscriminately to every row within a window.

Threshold-calibration, validation, and attack scores must retain their respective partition identities and timestamp alignment.

## 9. Score Artifact Requirements

The required per-observation or per-sequence score arrays have not all been persisted as artifacts. Previous notebook outputs and in-memory arrays must not be treated as saved files unless their existence is verified.

Notebook 08 will generate and persist the scores required for the evaluation using the frozen baseline definitions and the approved learned-model checkpoints.

This is an inference and scoring operation; it does not, by itself, constitute model training or retraining.

Before writing any new artifact, the notebook must verify the existing files and avoid overwriting model checkpoints or unrelated outputs.

Each saved score artifact must identify:

- the method;
- the data partition;
- the score definition;
- the relevant endpoint indices or timestamps;
- the feature set and preprocessing used;
- the model checkpoint or baseline configuration;
- any limitations affecting interpretation.

Score arrays must be checked for expected dimensions, finite values, and alignment with their corresponding observations or sequence endpoints.

The artifact inventory must distinguish between verified existing files, newly generated score files, and outputs that remain unavailable.

## 10. USAD Checkpoint Limitation

The current USAD checkpoint records `best_epoch = 1`.

The training procedure used:

$$
\alpha = \frac{1}{\text{epoch}},
\qquad
\beta = 1-\alpha
$$

At epoch 1:

$$
\alpha=1,\qquad \beta=0
$$

Therefore, the adversarial term involving the third reconstruction was inactive during that epoch's training.

The saved checkpoint consequently does not demonstrate successful training under an active adversarial-loss contribution. Its reconstruction scores can be evaluated, but the checkpoint's training limitation must be stated explicitly when interpreting its results.

The initial evaluation must use the verified existing checkpoint. Any later decision to tune or retrain USAD must follow the configuration audit and be documented as a separate experiment. Results from any revised checkpoint must not be attributed to the existing checkpoint.

## 11. Checkpoint Loading Requirements

The existing TCN and USAD checkpoint files have different structures.

The TCN checkpoint is a bare model state dictionary. The USAD checkpoint is a dictionary containing the model state dictionary and associated metadata.

The loading logic must respect these formats.

For TCN:

```python
tcn_state_dict = torch.load(
    tcn_checkpoint_path,
    map_location=device,
    weights_only=True,
)
tcn_model.load_state_dict(tcn_state_dict)
```

For USAD:

```python
usad_checkpoint = torch.load(
    usad_checkpoint_path,
    map_location=device,
    weights_only=True,
)
usad_model.load_state_dict(
    usad_checkpoint["model_state_dict"]
)
```

These examples assume a compatible PyTorch version and that the saved objects contain only weights-compatible data. The existing checkpoint structure must be verified before loading; the code must not be used to infer or alter the checkpoint format.

The USAD checkpoint metadata must also be inspected to confirm the saved configuration and best epoch.

## 12. Evaluation Metrics and Interpretation

After model configurations and thresholds have been fixed, evaluate the methods on the designated normal validation and attack datasets.

Where the score alignment and label construction permit, report:

- true positives (TP);
- false positives (FP);
- true negatives (TN);
- false negatives (FN);
- precision;
- recall;
- F1-score;
- false-positive rate;
- ROC-AUC and precision–recall AUC when valid continuous scores and both classes are available.

Confusion-matrix counts and derived metrics must use explicitly identified evaluation observations. The report must state how sequence endpoints, missing or unscorable values, and the two data sources are handled.

ROC-AUC and precision–recall AUC must be calculated from continuous anomaly scores, not from binary predictions alone.

Metric definitions, positive-class conventions, score direction, and any exclusions must be recorded. Undefined metrics must be reported as undefined rather than replaced with fabricated values.

Normal-validation flag rates must not be described as attack-detection rates. No method should be declared superior solely from a single metric without considering the evaluation limitations and differences in input features.

All final comparisons must state that the normal and attack datasets have overlapping wall-clock ranges and therefore do not form a temporally independent holdout.

## 13. Execution Order

The work in this notebook will proceed in the following order:

1. Record and verify the evaluation protocol.
2. Audit the existing configurations and checkpoint formats of all six learned models.
3. Decide and document whether any learned-model configuration tuning is justified.
4. If tuning is approved, conduct bounded experiments using development data only and preserve existing artifacts.
5. Freeze the model configurations to be evaluated.
6. Generate and persist the required normal threshold-calibration, normal validation, and attack scores.
7. Validate score dimensions, finiteness, endpoint alignment, and artifact integrity.
8. Calibrate learned-model thresholds using normal threshold-calibration scores only.
9. Freeze all thresholds before examining final evaluation performance.
10. Evaluate the nine methods and calculate the supported metrics.
11. Document results, limitations, and unresolved issues before proceeding to error analysis.

The order must not be reversed to accommodate a promising result. If a checkpoint or score artifact fails verification, document the failure and resolve it before using that artifact for evaluation.

## 14. Reporting Principles

The final report must distinguish:

- observed results from hypotheses;
- empirical sensor reference limits from industrial safety limits;
- anomaly scores from binary detection decisions;
- normal-validation false-positive behaviour from attack-detection performance;
- configuration tuning from threshold calibration;
- exact-timestamp separation from temporal independence;
- results obtained using 51-feature baselines from results obtained using 42-feature learned models;
- the current USAD checkpoint's performance from any results produced by a subsequently tuned or retrained version.

No result will be invented, inferred without evidence, or presented as proof of a confirmed cyberattack.

The objective is a reproducible and technically defensible comparison of the nine established methods, with limitations stated clearly enough that another analyst can reproduce the evaluation and understand what the evidence does—and does not—support.


In [29]:
from pathlib import Path

# PROJECT_ROOT must point to the SWATS project directory.
MODEL_DIR = PROJECT_ROOT / "swat-artifacts"

required_artifacts = {
    "Fitted feature scaler":
        MODEL_DIR / "scaler.joblib",

    "Normal evaluation sequences":
        MODEL_DIR / "evaluation_sequences.npz",

    "Attack evaluation sequences":
        MODEL_DIR / "attack_evaluation_sequences.npz",

    "Traditional ML model metadata":
        MODEL_DIR / "traditional_ml_model_metadata.json",

    "Isolation Forest checkpoint":
        MODEL_DIR / "isolation_forest.joblib",

    "SGD One-Class SVM checkpoint":
        MODEL_DIR / "sgd_one_class_svm.joblib",

    "LOF checkpoint":
        MODEL_DIR / "lof_model.joblib",

    "LSTM Autoencoder checkpoint":
        MODEL_DIR / "lstm_autoencoder.pth",

    "TCN Autoencoder checkpoint":
        MODEL_DIR / "tcn_preliminary" / "tcn_autoencoder_best.pth",

    "USAD checkpoint":
        MODEL_DIR / "usad_preliminary" / "usad_best.pth",

    "Traditional ML scores":
        MODEL_DIR / "traditional_ml_scores.npz",

    "LSTM Autoencoder scores":
        MODEL_DIR / "lstm_autoencoder_scores.npz",

    "TCN Autoencoder scores":
        MODEL_DIR / "tcn_autoencoder_scores.npz",

    "USAD preliminary scores":
        MODEL_DIR / "usad_preliminary_scores.npz",
}

# print("PROJECT_ROOT:", PROJECT_ROOT)
# print("MODEL_DIR:", MODEL_DIR)
print("\nARTIFACT VERIFICATION")
print("=" * 65)

missing_artifacts = []

for name, path in required_artifacts.items():
    if path.is_file():
        print(f"[FOUND]     {name}: {path.relative_to(PROJECT_ROOT)}")
    else:
        print(f"[NOT FOUND] {name}: {path.relative_to(PROJECT_ROOT)}")
        missing_artifacts.append(name)

print("\n" + "=" * 65)
print("Artifacts checked:", len(required_artifacts))
print("Artifacts found:", len(required_artifacts) - len(missing_artifacts))
print("Artifacts missing:", len(missing_artifacts))

if missing_artifacts:
    print("\nMissing artifacts:")
    for name in missing_artifacts:
        print("-", name)
else:
    print("\nAll listed artifacts are present.")


ARTIFACT VERIFICATION
[FOUND]     Fitted feature scaler: swat-artifacts/scaler.joblib
[FOUND]     Normal evaluation sequences: swat-artifacts/evaluation_sequences.npz
[FOUND]     Attack evaluation sequences: swat-artifacts/attack_evaluation_sequences.npz
[FOUND]     Traditional ML model metadata: swat-artifacts/traditional_ml_model_metadata.json
[FOUND]     Isolation Forest checkpoint: swat-artifacts/isolation_forest.joblib
[FOUND]     SGD One-Class SVM checkpoint: swat-artifacts/sgd_one_class_svm.joblib
[FOUND]     LOF checkpoint: swat-artifacts/lof_model.joblib
[FOUND]     LSTM Autoencoder checkpoint: swat-artifacts/lstm_autoencoder.pth
[FOUND]     TCN Autoencoder checkpoint: swat-artifacts/tcn_preliminary/tcn_autoencoder_best.pth
[FOUND]     USAD checkpoint: swat-artifacts/usad_preliminary/usad_best.pth
[FOUND]     Traditional ML scores: swat-artifacts/traditional_ml_scores.npz
[FOUND]     LSTM Autoencoder scores: swat-artifacts/lstm_autoencoder_scores.npz
[FOUND]     TCN Autoencod

In [30]:
import json
import joblib
from pathlib import Path


#  Read saved traditional ML metadata

metadata_path = MODEL_DIR / "traditional_ml_model_metadata.json"

with metadata_path.open("r", encoding="utf-8") as file:
    traditional_metadata = json.load(file)

print("TRADITIONAL ML MODEL METADATA")
print("=" * 70)
print(json.dumps(traditional_metadata, indent=2, default=str))



# Load existing model artifacts without fitting them

model_paths = {
    "Isolation Forest":
        MODEL_DIR / "isolation_forest.joblib",

    "SGD One-Class SVM":
        MODEL_DIR / "sgd_one_class_svm.joblib",

    "Local Outlier Factor":
        MODEL_DIR / "lof_model.joblib",
}

traditional_models = {}

print("\n\nLOADED MODEL CONFIGURATIONS")
print("=" * 70)

for model_name, model_path in model_paths.items():
    model = joblib.load(model_path)
    traditional_models[model_name] = model

    print(f"\n{model_name}")
    print("-" * 50)
    print("Artifact:", model_path.relative_to(PROJECT_ROOT))
    print("Class:", type(model).__name__)

    if hasattr(model, "get_params"):
        params = model.get_params(deep=False)

        for key, value in params.items():
            print(f"{key}: {value}")

    else:
        print("Warning: get_params() is unavailable for this artifact.")

TRADITIONAL ML MODEL METADATA
{
  "notebook": "06_traditional_ml_anomaly_detection",
  "scikit_learn_version": "1.6.1",
  "feature_count": 42,
  "feature_columns": [
    "FIT101",
    "LIT101",
    "MV101",
    "P101",
    "P102",
    "AIT201",
    "AIT202",
    "AIT203",
    "FIT201",
    "MV201",
    "P201",
    "P203",
    "P205",
    "DPIT301",
    "FIT301",
    "LIT301",
    "MV301",
    "MV302",
    "MV303",
    "MV304",
    "P302",
    "AIT401",
    "AIT402",
    "FIT401",
    "LIT401",
    "P402",
    "P403",
    "UV401",
    "AIT501",
    "AIT502",
    "AIT503",
    "AIT504",
    "FIT501",
    "FIT502",
    "FIT503",
    "FIT504",
    "P501",
    "PIT501",
    "PIT502",
    "PIT503",
    "FIT601",
    "P602"
  ],
  "training_observations": 221366,
  "training_features": 42,
  "models": {
    "isolation_forest": {
      "artifact": "isolation_forest.joblib",
      "n_estimators": 200,
      "max_samples": "auto",
      "contamination": "auto",
      "random_state": 42
    },
  

### 6.1 Traditional Machine-Learning Configuration Audit

The saved Isolation Forest, SGD One-Class SVM, and Local Outlier Factor artifacts were loaded successfully and inspected without fitting or modifying the models.

The audit confirmed the following configurations:

| Model | Verified settings |
|---|---|
| Isolation Forest | `n_estimators=200`, `max_samples="auto"`, `contamination="auto"`, `random_state=42` |
| SGD One-Class SVM | `nu=0.01`, `max_iter=80`, `tol=None`, `random_state=42`, `learning_rate="optimal"` |
| LOF | `n_neighbors=20`, `novelty=True`, `metric="minkowski"`, `p=2`, `contamination="auto"` |

The models use the established 42-feature set and were trained on 221,366 observations, as recorded in the traditional ML metadata.

The saved configurations are available for subsequent evaluation. No model was retrained or tuned during this audit.

Two issues remain relevant to reproducibility and the tuning decision:

1. The traditional ML metadata JSON records the Isolation Forest and SGD One-Class SVM configurations but does not include the LOF configuration. The LOF parameters were recovered directly from the saved artifact.
2. The metadata records scikit-learn version 1.6.1, whereas the audit environment has been reported as version 1.9.1. Successful artifact loading does not establish complete cross-version equivalence. This compatibility issue must be documented and addressed in the reproducibility stage.

The SGD One-Class SVM's iteration limit and stopping configuration warrant review against its existing training record before any tuning decision is made. The available configuration alone does not establish that the model failed to converge.

**Audit status:** Existing configurations verified. No tuning decision has been made.

In [31]:
import json
import torch

print("PYTORCH ENVIRONMENT")
print("=" * 60)
print("PyTorch version:", torch.__version__)
print("Device available:", "CUDA" if torch.cuda.is_available() else "CPU")

# ---------------------------------------------------------
# LSTM: inspect the saved checkpoint structure
# ---------------------------------------------------------
lstm_path = MODEL_DIR / "lstm_autoencoder.pth"

lstm_checkpoint = torch.load(
    lstm_path,
    map_location="cpu",
    weights_only=True,
)

print("\nLSTM CHECKPOINT")
print("=" * 60)
print("Checkpoint type:", type(lstm_checkpoint).__name__)

if isinstance(lstm_checkpoint, dict):
    print("Checkpoint keys:", list(lstm_checkpoint.keys()))

    for key, value in lstm_checkpoint.items():
        if isinstance(value, torch.Tensor):
            print(f"{key}: tensor shape {tuple(value.shape)}")
        elif isinstance(value, (int, float, str, bool, type(None))):
            print(f"{key}: {value}")


# ---------------------------------------------------------
# TCN: inspect saved training metadata
# ---------------------------------------------------------
tcn_metadata_path = (
    MODEL_DIR / "tcn_preliminary" / "tcn_training_metadata.json"
)

print("\nTCN TRAINING METADATA")
print("=" * 60)

if tcn_metadata_path.is_file():
    with tcn_metadata_path.open("r", encoding="utf-8") as file:
        tcn_metadata = json.load(file)

    print(json.dumps(tcn_metadata, indent=2, default=str))
else:
    print("TCN training metadata file not found.")


# ---------------------------------------------------------
# USAD: inspect checkpoint configuration
# ---------------------------------------------------------
usad_path = MODEL_DIR / "usad_preliminary" / "usad_best.pth"

usad_checkpoint = torch.load(
    usad_path,
    map_location="cpu",
    weights_only=True,
)

print("\nUSAD CHECKPOINT")
print("=" * 60)
print("Checkpoint type:", type(usad_checkpoint).__name__)

if isinstance(usad_checkpoint, dict):
    print("Checkpoint keys:", list(usad_checkpoint.keys()))

    if "config" in usad_checkpoint:
        print("\nSaved configuration:")
        print(json.dumps(
            usad_checkpoint["config"],
            indent=2,
            default=str,
        ))

    if "best_epoch" in usad_checkpoint:
        print("Best epoch:", usad_checkpoint["best_epoch"])

    if "best_monitor_reconstruction_mse" in usad_checkpoint:
        print(
            "Best monitoring reconstruction MSE:",
            usad_checkpoint["best_monitor_reconstruction_mse"],
        )

    if "model_state_dict" in usad_checkpoint:
        print(
            "State-dictionary tensor count:",
            len(usad_checkpoint["model_state_dict"]),
        )

PYTORCH ENVIRONMENT
PyTorch version: 2.11.0+cu128
Device available: CUDA

LSTM CHECKPOINT
Checkpoint type: OrderedDict
Checkpoint keys: ['encoder.weight_ih_l0', 'encoder.weight_hh_l0', 'encoder.bias_ih_l0', 'encoder.bias_hh_l0', 'encoder.weight_ih_l1', 'encoder.weight_hh_l1', 'encoder.bias_ih_l1', 'encoder.bias_hh_l1', 'encoder_projection.weight', 'encoder_projection.bias', 'decoder_projection.weight', 'decoder_projection.bias', 'decoder.weight_ih_l0', 'decoder.weight_hh_l0', 'decoder.bias_ih_l0', 'decoder.bias_hh_l0', 'decoder.weight_ih_l1', 'decoder.weight_hh_l1', 'decoder.bias_ih_l1', 'decoder.bias_hh_l1', 'output_layer.weight', 'output_layer.bias']
encoder.weight_ih_l0: tensor shape (256, 42)
encoder.weight_hh_l0: tensor shape (256, 64)
encoder.bias_ih_l0: tensor shape (256,)
encoder.bias_hh_l0: tensor shape (256,)
encoder.weight_ih_l1: tensor shape (256, 64)
encoder.weight_hh_l1: tensor shape (256, 64)
encoder.bias_ih_l1: tensor shape (256,)
encoder.bias_hh_l1: tensor shape (256,)

### 6.2 Deep-Learning Configuration Audit

#### Objective

Inspect the saved LSTM Autoencoder, TCN Autoencoder, and USAD artifacts to verify their recorded architectures and available training configurations before making any model-tuning decisions.

The audit uses existing checkpoints and metadata. It does not retrain models, regenerate scores, or modify saved artifacts.

#### LSTM Autoencoder

The saved LSTM checkpoint was loaded successfully. Its state dictionary contains the encoder and decoder LSTM parameters, the encoder and decoder projection layers, and the output layer.

The parameter dimensions confirm an architecture with 42 input features, a 64-unit hidden representation, two LSTM layers, a 16-dimensional bottleneck, and a 42-feature reconstruction output. The model operates on sequences of 30 observations.

The inspected checkpoint contains model parameters only. Training settings and training history therefore require supporting notebook records or metadata for independent verification.

**Audit finding:** The saved parameter dimensions are consistent with the intended LSTM Autoencoder architecture. No architectural change is justified by this inspection alone.

#### TCN Autoencoder

The saved training metadata confirms the following configuration:

- Input features: 42
- Sequence length: 30 observations
- Channels: `[32, 32, 32, 32]`
- Kernel size: 3
- Dilations: `[1, 2, 4, 8]`
- Convolutions per block: 2
- Recorded receptive field: 61 observations
- Trainable parameters: 40,554
- Batch size: 256
- Maximum epochs: 50
- Completed epochs: 46
- Best epoch: 41
- Initial learning rate: 0.001
- Early-stopping patience: 5
- Checkpoint-selection metric: monitoring MSE

The best recorded monitoring MSE was approximately 0.00057709, and the corresponding monitoring MAE was approximately 0.01316245.

The metadata records a checkpoint path from the original Google Drive environment. The locally verified artifact remains the relevant checkpoint for this project.

**Audit finding:** The architecture and recorded training configuration are documented. The relationship between the 61-observation receptive field and the 30-observation input window should be retained as a design consideration, not treated as proof of a defect.

#### USAD

The saved USAD checkpoint was loaded successfully. It contains a model state dictionary, the best epoch, the best monitoring reconstruction MSE, and a configuration dictionary.

The recorded configuration is:

- Input features: 42
- Sequence length: 30 observations
- Flattened input dimension: 1,260
- Hidden dimension: 128
- Latent dimension: 32
- Batch size: 256
- Maximum epochs: 50
- Learning rate: 0.0001
- Gradient clipping norm: 1.0
- Early-stopping patience: 5
- Best epoch: 1

The best recorded monitoring reconstruction MSE was approximately 0.02222458.

The existing training record indicates that monitoring reconstruction error increased substantially after epoch 1. At the best epoch, the recorded loss schedule had alpha equal to 1 and beta equal to 0, leaving the beta-weighted loss term inactive at that epoch.

**Audit finding:** The saved checkpoint and its configuration are available, but the training outcome warrants further investigation. The preliminary result does not establish successful adversarial training. No retraining or parameter change is authorized by this audit alone.

#### Audit conclusion

The LSTM and TCN configurations are supported by their inspected artifacts and available metadata. USAD has a documented training concern that must be considered before its final configuration is approved.

This audit does not change any model, threshold, feature set, or saved score array. Decisions about tuning will be made only after the configuration audits for all six learned models have been considered together.

### 7. Hyperparameter Tuning Plan

#### 7.1 Objective

Tune the six learned anomaly-detection models using controlled experiments to investigate whether alternative configurations improve their suitability for the SWaT anomaly-detection task.

The models are Isolation Forest, SGD One-Class SVM, Local Outlier Factor (LOF), LSTM Autoencoder, TCN Autoencoder, and USAD.

The existing models, checkpoints, and score arrays will be preserved until candidate configurations have been trained, evaluated using permitted data, and verified.

#### 7.2 Data Separation

The tuning process will respect the existing chronological partitions:

| Partition | Observations | Permitted use |
|---|---:|---|
| Training | 221,366 | Fit candidate models |
| Monitoring | 47,435 | Compare candidate configurations and support model selection |
| Threshold calibration | 47,437 | Calibrate thresholds after model configurations have been selected |
| Normal validation | 79,060 | Final evaluation only |
| Attack evaluation | 54,621 | Final evaluation only |

The monitoring partition contains normal observations. Consequently, monitoring reconstruction error and score stability can provide useful evidence, but they cannot establish attack-detection performance.

Attack labels and final-validation results must not influence hyperparameter selection. Threshold-calibration observations must not be used to select model configurations.


#### 7.3 Hyperparameter Search Spaces

The search spaces for the six learned models will be finalized using evidence from their existing configurations, training histories, monitoring results, computational requirements, and implementation details.

Candidate values must have a documented technical justification. The initial values recorded in this section are provisional and must not be treated as an approved search space until the model-specific review is complete.

Existing checkpoints, score arrays, preprocessing artifacts, and training records will be preserved. No tuning experiment will begin until its search space, selection criteria, and execution procedure have been reviewed.

**Model 1 — Isolation Forest**

The existing configuration uses 200 estimators, automatic maximum samples, all available features, and random seed 42. The model was fitted using 221,366 training observations and 42 features.

The initial search will examine the following parameters:

| Hyperparameter | Candidate values |
|---|---|
| Number of estimators (`n_estimators`) | 100, 200, 300 |
| Maximum samples (`max_samples`) | 256, 1,024, 4,096 |
| Maximum features (`max_features`) | 0.75, 1.0 |
| Random seed | 42 |

The search contains 18 candidate configurations. The maximum-sample candidates examine the effect of changing the number of observations used to construct individual trees. The estimator-count candidates examine score-ranking stability against computational cost. The feature candidates examine whether feature subsampling changes the resulting rankings.

The `contamination` parameter will not be tuned because the project uses a separate threshold-calibration stage. Candidate models will be compared using anomaly-score ranking stability on the monitoring partition and recorded computational costs.

Since the monitoring partition contains normal observations only, these comparisons cannot establish attack-detection superiority. The selected configuration and its limitations must be documented before proceeding to the next model.

The existing model, checkpoint-related artifacts, and saved score arrays must remain unchanged during the search.

**Model 2 — SGD One-Class SVM**

| Hyperparameter | Candidate values |
|---|---|
| Regularization (`alpha`) | 0.0001, 0.001, 0.01 |
| Anomaly fraction parameter (`nu`) | 0.005, 0.01, 0.05 |
| Maximum iterations (`max_iter`) | 80, 200, 500 |
| Random seed | 42 |

The existing configuration uses `nu=0.01`, `max_iter=80`, and `tol=None`. Candidate experiments must record convergence-related warnings, iteration limits, and execution time. The search must not assume that increasing the iteration limit necessarily improves detection.

**Model 3 — Local Outlier Factor**

| Hyperparameter | Candidate values |
|---|---|
| Number of neighbors (`n_neighbors`) | 10, 20, 35 |
| Novelty detection | `True` |
| Distance metric | Minkowski, with `p=2` |
| Algorithm | `auto` |

LOF must remain in novelty-detection mode because observations are scored after fitting on the training data. The experiment will record fitting and scoring time as well as score stability.

**Model 4 — LSTM Autoencoder**

The existing architecture uses 42 features, 30-observation sequences, two LSTM layers, a hidden size of 64, and a bottleneck size of 16.

The initial investigation will consider:

| Hyperparameter | Candidate values |
|---|---|
| Hidden size | 32, 64 |
| Bottleneck dimension | 8, 16 |
| Learning rate | 0.0001, 0.001 |
| Batch size | 256 |
| Random seed | 42 |

The number of layers and sequence length will remain fixed during this initial search. This limits computational cost and makes the effects of the selected hyperparameters easier to interpret.

Candidate models will be fitted on training sequences and compared using monitoring reconstruction error and training stability.

**Model 5 — TCN Autoencoder**

The existing architecture uses 42 features, 30-observation sequences, four convolutional blocks, kernel size 3, and dilations `[1, 2, 4, 8]`.

The initial investigation will consider:

| Hyperparameter | Candidate values |
|---|---|
| Channel configuration | `[16, 16, 16, 16]`, `[32, 32, 32, 32]` |
| Learning rate | 0.0001, 0.001 |
| Batch size | 256 |
| Random seed | 42 |

The sequence length, kernel size, dilation schedule, and number of blocks will remain fixed initially. The recorded receptive field and parameter count will be checked for each candidate architecture.

Candidate models will be compared using monitoring reconstruction error, training stability, parameter count, and execution time.

**Model 6 — USAD**

The existing architecture uses 30-observation sequences, 42 features, a flattened input dimension of 1,260, a hidden dimension of 128, and a latent dimension of 32.

The initial investigation will consider:

| Hyperparameter | Candidate values |
|---|---|
| Hidden dimension | 64, 128 |
| Latent dimension | 16, 32 |
| Learning rate | 0.00001, 0.0001 |
| Batch size | 256 |
| Maximum epochs | 50 |
| Random seed | 42 |

The training procedure, both optimizer updates, loss schedule, and monitoring logic must be checked before interpreting candidate results. The existing checkpoint's best epoch and subsequent monitoring instability must be documented.

Candidate configurations will be assessed for finite losses, stable training, monitoring reconstruction error, and execution time. A low reconstruction error alone will not establish successful adversarial training or effective anomaly detection.

#### 7.4 Model Selection and Evidence

For the autoencoders, monitoring reconstruction error will be recorded alongside training stability. The selected checkpoint must be determined using the monitoring partition, not final evaluation data.

For Isolation Forest, SGD One-Class SVM, and LOF, score distributions, ranking stability, fitting behavior, and computational cost will be examined. Since the monitoring data contain normal observations only, these diagnostics cannot independently establish which configuration has the best attack-detection performance.

A configuration will not be selected solely because it produces the lowest anomaly scores or the fewest normal alerts. Any selection must have a documented technical justification. Where the available evidence cannot distinguish candidates defensibly, that limitation will be recorded.

#### 7.5 Experimental Controls

All experiments must follow these controls:

1. Preserve the existing training, monitoring, threshold-calibration, and evaluation partitions.
2. Fit preprocessing transformations on training data only.
3. Keep the learned models' 42-feature input representation unchanged during the initial tuning experiments.
4. Use the same designated monitoring partition for comparable candidate evaluations.
5. Fix and record random seeds.
6. Record every candidate configuration, training outcome, monitoring result, warning, and execution time.
7. Preserve the original checkpoints and score arrays until replacement artifacts have been verified.
8. Do not tune thresholds during hyperparameter selection.
9. Do not use attack labels or final-validation results to select configurations.
10. Reuse existing results wherever possible; do not retrain a model unless the experiment requires a new candidate configuration.

#### 7.6 Execution Order

Tuning will proceed in the following order:

1. Isolation Forest
2. SGD One-Class SVM
3. LOF
4. LSTM Autoencoder
5. TCN Autoencoder
6. USAD

Each model's experiment and results must be documented before moving to the next model. The search spaces may be refined only when the evidence, implementation constraints, or computational cost justify a documented change.

After all six configuration searches are complete, the selected configurations will be frozen. Threshold calibration and final evaluation will then proceed as separate stages.

In [32]:
import numpy as np

SCORES_PATH = MODEL_DIR / "traditional_ml_scores.npz"

if not SCORES_PATH.is_file():
    raise FileNotFoundError(
        f"Existing traditional ML score archive not found: {SCORES_PATH}"
    )

print("TRADITIONAL ML SCORE ARCHIVE")
print("=" * 60)
# print("File:", SCORES_PATH)
print(f"Size: {SCORES_PATH.stat().st_size:,} bytes")

with np.load(SCORES_PATH, allow_pickle=False) as scores:
    print("\nStored arrays:")

    for name in scores.files:
        array = scores[name]
        print(
            f"{name}: "
            f"shape={array.shape}, "
            f"dtype={array.dtype}"
        )

TRADITIONAL ML SCORE ARCHIVE
Size: 1,708,436 bytes

Stored arrays:
if_threshold_scores: shape=(47437,), dtype=float64
if_validation_scores: shape=(79060,), dtype=float64
ocsvm_threshold_scores: shape=(47437,), dtype=float64
ocsvm_validation_scores: shape=(79060,), dtype=float64
lof_threshold_scores: shape=(47437,), dtype=float32
lof_validation_scores: shape=(79060,), dtype=float32


In [33]:
print("MONITORING DATA CHECK")
print("=" * 60)

for name in (
    "X_train",
    "X_monitor",
    "X_train_scaled",
    "X_monitor_scaled",
):
    if name in globals():
        value = globals()[name]
        print(
            f"{name}: "
            f"type={type(value).__name__}, "
            f"shape={getattr(value, 'shape', 'not available')}"
        )
    else:
        print(f"{name}: not defined in the current kernel")

MONITORING DATA CHECK
X_train: not defined in the current kernel
X_monitor: not defined in the current kernel
X_train_scaled: not defined in the current kernel
X_monitor_scaled: not defined in the current kernel


In [34]:
print("EXISTING DATA AND PREPROCESSING VARIABLES")
print("=" * 60)

keywords = (
    "train", "monitor", "threshold", "validation",
    "scaled", "sequence", "feature", "split", "normal"
)

for name, value in sorted(globals().items()):
    if any(word in name.lower() for word in keywords):
        if name.startswith("_"):
            continue

        shape = getattr(value, "shape", None)
        if shape is not None:
            print(f"{name}: {type(value).__name__}, shape={shape}")
        elif isinstance(value, (str, int, float, bool)):
            print(f"{name}: {type(value).__name__} = {value}")
        elif isinstance(value, (list, tuple)):
            print(f"{name}: {type(value).__name__}, length={len(value)}")

EXISTING DATA AND PREPROCESSING VARIABLES
EXCLUDED_FEATURES: list, length=9
FEATURE_COLUMNS: list, length=42
MONITOR_ROWS: int = 47435
THRESHOLD_ROWS: int = 47437
TRAIN_ROWS: int = 221366
VALIDATION_ROWS: int = 79060
X_attack_scaled: ndarray, shape=(54621, 42)
X_threshold: ndarray, shape=(47437, 42)
X_threshold_raw: DataFrame, shape=(47437, 42)
X_threshold_seq: ndarray, shape=(47408, 30, 42)
X_validation: ndarray, shape=(79060, 42)
X_validation_raw: DataFrame, shape=(79060, 42)
X_validation_seq: ndarray, shape=(79031, 30, 42)
attack_sequence_labels: Series, shape=(54592,)
attack_sequence_timestamps: Series, shape=(54592,)
attack_sequences: ndarray, shape=(54592, 30, 42)
eligible_normal: DataFrame, shape=(395298, 53)
expected_threshold_sequences: int = 47408
expected_validation_sequences: int = 79031
if_threshold_scores: ndarray, shape=(47437,)
if_threshold_seq_scores: ndarray, shape=(47408,)
if_validation_scores: ndarray, shape=(79060,)
if_validation_seq_scores: ndarray, shape=(79031,)

In [35]:
print("VERIFY EXISTING NORMAL-DATA PARTITIONS")
print("=" * 65)

timestamp_col = "Timestamp"

print(f"Eligible normal observations: {len(eligible_normal):,}")
print(f"Expected total: {TRAIN_ROWS + MONITOR_ROWS + THRESHOLD_ROWS + VALIDATION_ROWS:,}")
print(f"Chronological order: {eligible_normal[timestamp_col].is_monotonic_increasing}")

partitions = [
    ("Training", 0, TRAIN_ROWS),
    ("Monitoring", TRAIN_ROWS, TRAIN_ROWS + MONITOR_ROWS),
    (
        "Threshold calibration",
        TRAIN_ROWS + MONITOR_ROWS,
        TRAIN_ROWS + MONITOR_ROWS + THRESHOLD_ROWS,
    ),
    (
        "Validation",
        TRAIN_ROWS + MONITOR_ROWS + THRESHOLD_ROWS,
        len(eligible_normal),
    ),
]

for name, start, end in partitions:
    partition = eligible_normal.iloc[start:end]

    print(f"\n{name}")
    print(f"  Expected rows: {end - start:,}")
    print(f"  Actual rows:   {len(partition):,}")
    print(f"  First timestamp: {partition[timestamp_col].iloc[0]}")
    print(f"  Last timestamp:  {partition[timestamp_col].iloc[-1]}")

VERIFY EXISTING NORMAL-DATA PARTITIONS
Eligible normal observations: 395,298
Expected total: 395,298
Chronological order: True

Training
  Expected rows: 221,366
  Actual rows:   221,366
  First timestamp: 2015-12-28 10:00:00
  Last timestamp:  2015-12-31 12:43:42

Monitoring
  Expected rows: 47,435
  Actual rows:   47,435
  First timestamp: 2015-12-31 12:43:43
  Last timestamp:  2016-01-01 02:23:17

Threshold calibration
  Expected rows: 47,437
  Actual rows:   47,437
  First timestamp: 2016-01-01 02:23:18
  Last timestamp:  2016-01-01 15:51:19

Validation
  Expected rows: 79,060
  Actual rows:   79,060
  First timestamp: 2016-01-01 15:51:20
  Last timestamp:  2016-01-02 14:59:59


In [36]:
# Recover the established chronological partitions.
# This does not modify eligible_normal or create a new split.

train_end = TRAIN_ROWS
monitor_end = TRAIN_ROWS + MONITOR_ROWS

train_df = eligible_normal.iloc[:train_end].copy()
monitor_df = eligible_normal.iloc[train_end:monitor_end].copy()

# Select the exact 42 features used by the learned models.
X_train_raw = train_df[FEATURE_COLUMNS].copy()
X_monitor_raw = monitor_df[FEATURE_COLUMNS].copy()

# Reuse the existing scaler fitted on training data only.
X_train_scaled = scaler.transform(X_train_raw)
X_monitor_scaled = scaler.transform(X_monitor_raw)

print("RECOVERED TRAINING AND MONITORING DATA")
print("=" * 60)

print(f"Training rows: {len(X_train_raw):,}")
print(f"Monitoring rows: {len(X_monitor_raw):,}")
print(f"Training feature shape: {X_train_scaled.shape}")
print(f"Monitoring feature shape: {X_monitor_scaled.shape}")
print(f"Feature count: {len(FEATURE_COLUMNS)}")

print("\nTimestamp boundaries")
print(f"Training: {train_df['Timestamp'].iloc[0]} to {train_df['Timestamp'].iloc[-1]}")
print(f"Monitoring: {monitor_df['Timestamp'].iloc[0]} to {monitor_df['Timestamp'].iloc[-1]}")

print("\nData checks")
print(f"Training features contain NaN: {np.isnan(X_train_scaled).any()}")
print(f"Monitoring features contain NaN: {np.isnan(X_monitor_scaled).any()}")
print(f"Training features contain infinity: {np.isinf(X_train_scaled).any()}")
print(f"Monitoring features contain infinity: {np.isinf(X_monitor_scaled).any()}")

RECOVERED TRAINING AND MONITORING DATA
Training rows: 221,366
Monitoring rows: 47,435
Training feature shape: (221366, 42)
Monitoring feature shape: (47435, 42)
Feature count: 42

Timestamp boundaries
Training: 2015-12-28 10:00:00 to 2015-12-31 12:43:42
Monitoring: 2015-12-31 12:43:43 to 2016-01-01 02:23:17

Data checks
Training features contain NaN: False
Monitoring features contain NaN: False
Training features contain infinity: False
Monitoring features contain infinity: False


#### Isolation Forest Candidate Selection

**Evaluation data**

The model is fitted using the established training partition of 221,366 observations and 42 features. The subsequent monitoring partition of 47,435 normal observations is used to examine score-distribution behaviour. The threshold-calibration and final-validation partitions are excluded from hyperparameter selection.

**Selection procedure**

Each of the 18 candidate configurations is evaluated using three fixed random seeds: 42, 43, and 44. These additional seeds are used to assess sensitivity to model randomness; they do not replace the project's primary reproducibility seed of 42.

Candidate evaluation considers the following:

1. **Ranking stability:** Calculate pairwise Spearman rank correlations between anomaly scores produced by the three seeds, separately for the training and monitoring observations. Summarize the correlations for each configuration using their median.
2. **Score-distribution behaviour:** Compare training and monitoring score distributions using descriptive statistics and distribution summaries. Changes are documented as potential differences between the observed normal partitions, not automatically interpreted as model failure.
3. **Computational efficiency:** Record fitting time for each candidate configuration and compare the additional computational cost of more complex configurations.
4. **Parameter sensitivity:** Examine how ensemble size, subsample size, and feature subsampling affect ranking stability and score behaviour.

**Selection rule**

The primary selection criterion is reproducibility of anomaly-score rankings across random seeds. Candidate configurations are compared using their median pairwise Spearman correlations, with training and monitoring results reported separately. Computational cost and parameter sensitivity inform the final decision when stability results are comparable.

No arbitrary score-stability cutoff is imposed in advance. The observed results, trade-offs, and rationale for the selected configuration are recorded explicitly.

This procedure evaluates the consistency of the anomaly rankings, not their ability to identify attacks. Since the monitoring partition contains normal observations only and no independent attack-detection criterion is available for model selection, the selected configuration must not be described as having superior attack-detection performance.

**Reproducibility and artifact preservation**

The existing Isolation Forest model and its saved scores remain unchanged during candidate evaluation. Each candidate's parameters, random seed, fitting time, and evaluation summaries are recorded separately. The selected configuration is documented before any final model replacement or threshold calibration is considered.

#### SGD One-Class SVM

**Objective**

Evaluate how the regularization parameter, optimization duration, and stopping configuration affect the consistency of anomaly-score rankings produced by `SGDOneClassSVM`.

**Search space**

| Hyperparameter | Candidate values | Rationale |
|---|---|---|
| `nu` | 0.001, 0.01, 0.05 | Examine the effect of the parameter controlling the upper bound on training errors and the lower bound on the fraction of support vectors in the One-Class SVM formulation. |
| `max_iter` | 80, 200, 500 | Assess whether additional optimization iterations materially change the fitted decision function. |
| `tol` | `None`, 0.001 | Compare a fixed iteration budget with tolerance-based stopping. |

The search contains 18 candidate configurations. The existing configuration (`nu=0.01`, `max_iter=80`, `tol=None`) is included.

Other optimization parameters remain fixed at their existing values during this search to isolate the effects of the selected hyperparameters. The random seed remains 42 for the primary reproducible fit. Additional seeds may be used to assess ranking stability after the initial candidate comparison.

**Evaluation procedure**

Each candidate is fitted using the established training partition. Anomaly scores are calculated for the training and monitoring partitions using a consistent score definition and orientation.

Candidate comparisons consider:

1. **Ranking stability:** Agreement between anomaly-score rankings across repeated random seeds, where evaluated.
2. **Optimization sensitivity:** Changes in the fitted decision function and anomaly-score rankings as the iteration budget and stopping configuration change.
3. **Monitoring behaviour:** Differences between training and monitoring score distributions, documented without treating distribution changes as definitive evidence of model failure.
4. **Computational cost:** Fit time and the additional cost of longer optimization runs.

The training and monitoring observations are used only for model development. Threshold-calibration and final-validation scores are excluded from hyperparameter selection.

**Selection and reporting**

The selected configuration is justified using the observed ranking stability, optimization sensitivity, monitoring behaviour, and computational cost. No configuration is declared superior at attack detection based solely on normal-only observations.

The candidate parameters, random seeds, fitting times, evaluation summaries, and selection rationale are recorded. The existing fitted model and score artifacts remain unchanged until a selected configuration has been documented and a separate decision has been made about updating the final model.

#### Local Outlier Factor (LOF)

**Objective**

Evaluate how neighborhood size and distance weighting affect the consistency of LOF anomaly-score rankings on the established training and monitoring partitions.

**Search space**

| Hyperparameter | Candidate values | Rationale |
|---|---|---|
| `n_neighbors` | 10, 20, 35 | Assess sensitivity to local neighborhood size. |
| `weights` | `uniform`, `distance` | Not applicable to LOF; this parameter is excluded from the search. |
| `metric` | `minkowski` | Retain the existing distance metric. |
| `p` | 2 | Retain Euclidean distance. |
| `novelty` | `True` | Required for scoring observations outside the fitted training data. |
| `contamination` | `"auto"` | Keep model fitting separate from threshold calibration. |

The initial search contains three candidate configurations. The existing configuration, `n_neighbors=20`, is included.

**Evaluation procedure**

Each candidate is fitted using the established training partition and evaluated on the training and monitoring partitions using a consistent anomaly-score definition and orientation.

The evaluation records:

1. **Ranking behaviour:** Compare anomaly-score rankings across neighborhood sizes using pairwise Spearman rank correlations.
2. **Score-distribution behaviour:** Summarize training and monitoring score distributions and document notable differences.
3. **Computational cost:** Record fitting time for each candidate.
4. **Neighborhood sensitivity:** Examine whether changes in neighborhood size materially affect the resulting anomaly rankings.

The threshold-calibration and final-validation partitions are excluded from hyperparameter selection. Because LOF is computationally expensive, repeated-seed experiments will be considered only if the initial results indicate that they are necessary to resolve uncertainty about ranking stability.

**Selection and reporting**

The selected neighborhood size is justified using ranking behaviour, monitoring results, computational cost, and sensitivity to the choice of neighborhood size. No candidate is declared superior at attack detection based on normal-only observations.

The candidate parameters, fitting times, evaluation summaries, and selection rationale are recorded. The existing LOF model and score artifacts remain unchanged until a selected configuration has been documented and a separate decision has been made about updating the final model.

#### LSTM Autoencoder

**Objective**

Evaluate how the latent representation size, recurrent hidden dimension, and learning rate affect reconstruction performance and the consistency of anomaly scores on normal observations.

**Existing configuration**

| Parameter | Existing value |
|---|---:|
| Sequence length | 30 observations |
| Input features | 42 |
| Hidden dimension | 64 |
| LSTM layers | 2 |
| Bottleneck dimension | 16 |
| Dropout | 0.2 |
| Learning rate | 0.001 |
| Batch size | 256 |
| Learning-rate scheduler | ReduceLROnPlateau |
| Early-stopping patience | 5 epochs |
| Random seed | 42 |

The existing architecture and checkpoint provide the reference configuration. The saved model and score artifacts will be preserved during candidate evaluation.

**Proposed search space**

| Hyperparameter | Candidate values | Rationale |
|---|---|---|
| Hidden dimension | 32, 64 | Compare a smaller recurrent representation with the existing configuration. |
| Bottleneck dimension | 8, 16 | Assess the effect of latent representation capacity. |
| Learning rate | 0.0003, 0.001 | Compare a lower learning rate with the existing configuration. |

The initial search contains eight configurations. The existing configuration is included.

Sequence length, number of LSTM layers, dropout, batch size, optimizer type, and loss function remain fixed during this initial search. This limits the search to parameters with a clear relationship to representation capacity and optimization.

**Training and evaluation procedure**

Each candidate is trained using the established training sequences. The monitoring partition is used for model-development decisions and early stopping. The threshold-calibration and final-validation partitions remain excluded from hyperparameter selection.

The evaluation records:

1. **Reconstruction performance:** Training and monitoring reconstruction losses, with particular attention to divergence and the gap between the two.
2. **Training stability:** Epochs completed, learning-rate changes, early-stopping epoch, and any numerical instability.
3. **Anomaly-score behaviour:** The distribution of reconstruction-error scores on normal monitoring sequences.
4. **Computational cost:** Training duration and the number of trainable parameters.

Candidates are compared using their monitoring reconstruction loss and training stability. Reconstruction loss alone is not interpreted as proof of superior attack detection.

**Selection and reporting**

The selected configuration is justified using monitoring reconstruction performance, the training–monitoring loss gap, optimization stability, and computational cost. The candidate parameters, training histories, stopping epochs, and selection rationale are recorded.

After the configuration is selected, threshold calibration is performed separately using the designated normal threshold-calibration partition. Final evaluation is performed only after the model configuration and threshold have been fixed.

The existing LSTM Autoencoder checkpoint and score artifacts remain unchanged during the search.

#### TCN Autoencoder

**Objective**

Evaluate how channel capacity, convolutional kernel size, and learning rate affect reconstruction performance, training stability, and anomaly-score behaviour.

**Existing configuration**

| Parameter | Existing value |
|---|---:|
| Input features | 42 |
| Sequence length | 30 observations |
| Encoder channels | `[32, 32, 32, 32]` |
| Kernel size | 3 |
| Encoder dilations | `[1, 2, 4, 8]` |
| Convolution padding | Causal |
| Best monitoring MSE | 0.00057709 |
| Best epoch | 41 |
| Early-stopping epoch | 46 |
| Random seed | 42 |

The existing architecture, checkpoint, and score artifacts provide the reference configuration.

**Proposed search space**

| Hyperparameter | Candidate values | Rationale |
|---|---|---|
| Channel width | 16, 32 | Compare a smaller network with the existing channel width. |
| Kernel size | 3, 5 | Assess the effect of convolutional kernel width. |
| Learning rate | 0.0003, 0.001 | Compare a lower learning rate with the existing training rate, subject to verification against the original training configuration. |

The initial search contains eight configurations. The existing architecture is represented by channel width 32, kernel size 3, and the verified existing learning rate.

The dilation schedule, number of encoder blocks, sequence length, loss function, optimizer type, and other training settings remain fixed during the initial search. The decoder must remain compatible with each candidate encoder, and causal padding must be retained.

**Training and evaluation procedure**

Each candidate is trained using the established training sequences. The monitoring partition supports model selection and early stopping. The threshold-calibration and final-validation partitions remain excluded from hyperparameter selection.

The evaluation records:

1. **Reconstruction performance:** Training and monitoring losses, including the difference between them.
2. **Training stability:** Epochs completed, early-stopping epoch, and any numerical instability.
3. **Anomaly-score behaviour:** Reconstruction-error distributions on normal monitoring sequences.
4. **Model complexity:** Trainable parameter count for each candidate architecture.
5. **Computational cost:** Training duration.

Candidates are compared using monitoring reconstruction loss, training stability, model complexity, and computational cost. Reconstruction performance on normal observations is not treated as proof of superior attack detection.

**Selection and reporting**

The selected configuration is justified using monitoring reconstruction performance, training stability, model complexity, and computational cost. The search results, training histories, selected parameters, and selection rationale are recorded.

After configuration selection, the designated normal threshold-calibration partition is used for threshold calibration. Final evaluation occurs only after the model configuration and threshold have been fixed.

Existing TCN checkpoints, training histories, and score artifacts remain unchanged during candidate evaluation.

### 7.6 USAD Hyperparameter Tuning

#### Objective

Evaluate whether changes to the learning rate, hidden-layer capacity, and latent representation improve USAD training stability and monitoring reconstruction performance while preserving its two-decoder architecture and adversarial training objective.

The preliminary training run stopped after six epochs, with the best checkpoint selected at epoch 1. Monitoring reconstruction loss increased substantially during subsequent epochs. The tuning procedure must therefore evaluate optimization stability alongside reconstruction performance.

#### Existing Configuration

| Parameter | Existing value |
|---|---:|
| Input features | 42 |
| Sequence length | 30 observations |
| Flattened input dimension | 1,260 |
| Hidden dimension | 128 |
| Latent dimension | 32 |
| Batch size | 256 |
| Learning rate | 0.0001 |
| Maximum epochs | 50 |
| Early-stopping patience | 5 epochs |
| Gradient clipping | 1.0 |
| Random seed | 42 |

The existing checkpoint and score artifacts will be retained as reference results. The original training history must also be preserved for comparison.

#### Hyperparameter Search Space

| Hyperparameter | Candidate values | Rationale |
|---|---|---|
| Learning rate | 0.00001, 0.00005, 0.0001 | Assess whether a lower learning rate improves optimization stability. |
| Hidden dimension | 64, 128 | Compare reduced and existing hidden-layer capacity. |
| Latent dimension | 16, 32 | Assess the effect of latent representation capacity. |

The initial search contains eight configurations. The existing learning rate, hidden dimension, and latent dimension are included as reference settings.

Sequence length, feature count, batch size, maximum epochs, early-stopping patience, gradient clipping, optimizer arrangement, loss definitions, and adversarial weighting schedule will remain fixed during the initial search, subject to verification against the existing implementation.

#### Training and Evaluation Procedure

Each configuration will use the established chronological training and monitoring partitions. The monitoring partition will support checkpoint selection and early stopping. The threshold-calibration partition and final validation partition will remain excluded from hyperparameter selection.

The following information will be recorded for every run:

1. **Configuration and reproducibility:** Hyperparameter values, random seed, and model configuration.
2. **Training stability:** Both USAD training objectives, monitoring reconstruction loss, gradient behavior, non-finite values, and the epoch at which training stops.
3. **Reconstruction performance:** Monitoring reconstruction mean squared error (MSE) and mean absolute error (MAE).
4. **Checkpoint selection:** Best epoch, checkpoint path, and corresponding monitoring reconstruction metrics.
5. **Computational cost:** Training duration, completed epochs, and early-stopping outcome.

The loss components, gradient behavior, and parameter updates must be examined together when assessing optimization stability. A negative value for the second USAD objective is not, by itself, evidence of an implementation error.

#### Training Stability and Selection Criteria

A configuration will be considered eligible for selection only when training remains numerically stable and the implemented adversarial optimization behaves consistently with the intended USAD formulation.

Among eligible configurations, monitoring reconstruction performance, training stability, and computational cost will inform selection. A lower reconstruction error alone will not justify selecting a configuration if its training behavior is unstable.

The original epoch-1 checkpoint will remain a preliminary reference, not evidence of successful adversarial training. During epoch 1, the adversarial weighting coefficient \(\beta\) is zero; therefore, the adversarial component does not contribute to the objective in that epoch.

#### Limitations

The preliminary run exhibited a substantial increase in monitoring reconstruction loss after epoch 1. Hyperparameter tuning must establish whether the candidate configurations can train stably before conclusions are drawn about USAD's anomaly-detection performance.

The initial search investigates learning rate and representation capacity while keeping the existing optimization procedure fixed. If instability persists across configurations, the training implementation and optimization procedure will require a separate, evidence-based review rather than an unsupported expansion of the search space.

No attack labels or final-validation results will be used to select hyperparameters. Anomaly-threshold calibration and final evaluation will be performed only after the model configuration has been selected.

In [37]:
# Verify the original Notebook 04 baseline calibration protocol.
# This is an audit of the existing definitions, not a change to the baselines.

from IPython.display import display

BASELINE_TIMESTAMP_COL = "Timestamp"
BASELINE_LABEL_COL = "Normal/Attack"

required_columns = {
    BASELINE_TIMESTAMP_COL,
    BASELINE_LABEL_COL,
}

missing_columns = required_columns.difference(eligible_normal.columns)

if missing_columns:
    raise KeyError(
        f"Required columns are missing from eligible_normal: "
        f"{sorted(missing_columns)}"
    )

# The original baselines used all 51 sensor features.
baseline_sensor_cols = [
    column
    for column in eligible_normal.columns
    if column not in required_columns
]

if len(baseline_sensor_cols) != 51:
    raise ValueError(
        f"Expected 51 baseline sensor features, "
        f"but found {len(baseline_sensor_cols)}."
    )

# Reconstruct the exact 80% calibration / 20% validation split.
n_eligible = len(eligible_normal)
calibration_end = int(n_eligible * 0.80)

baseline_calibration_check = eligible_normal.iloc[
    :calibration_end
].copy()

baseline_validation_check = eligible_normal.iloc[
    calibration_end:
].copy()

if n_eligible != 395_298:
    raise ValueError(
        f"Expected 395,298 eligible observations, found {n_eligible:,}. "
        "Stop and verify the existing data-preparation state."
    )

if len(baseline_calibration_check) != 316_238:
    raise AssertionError("Original calibration row count does not match.")

if len(baseline_validation_check) != 79_060:
    raise AssertionError("Original validation row count does not match.")

# Verify timestamps without changing eligible_normal.
calibration_timestamps = pd.to_datetime(
    baseline_calibration_check[BASELINE_TIMESTAMP_COL],
    errors="coerce",
)

validation_timestamps = pd.to_datetime(
    baseline_validation_check[BASELINE_TIMESTAMP_COL],
    errors="coerce",
)

if calibration_timestamps.isna().any() or validation_timestamps.isna().any():
    raise ValueError("Invalid timestamps detected in a baseline partition.")

if not calibration_timestamps.is_monotonic_increasing:
    raise ValueError("Baseline calibration partition is not chronological.")

if not validation_timestamps.is_monotonic_increasing:
    raise ValueError("Baseline validation partition is not chronological.")

if calibration_timestamps.max() >= validation_timestamps.min():
    raise AssertionError(
        "The original baseline calibration and validation periods "
        "are not strictly separated."
    )

# Recalculate reference statistics in separately named audit variables.
# These do not replace or modify the original Notebook 04 references.
baseline_rule_limits_check = pd.DataFrame({
    "lower_limit": baseline_calibration_check[
        baseline_sensor_cols
    ].quantile(0.01),

    "upper_limit": baseline_calibration_check[
        baseline_sensor_cols
    ].quantile(0.99),
})

baseline_simple_reference_check = pd.DataFrame({
    "calibration_mean": baseline_calibration_check[
        baseline_sensor_cols
    ].mean(),

    "calibration_std": baseline_calibration_check[
        baseline_sensor_cols
    ].std(ddof=1),
})

print("NOTEBOOK 04 BASELINE PROTOCOL — VERIFICATION")
print("=" * 55)

print(f"Eligible normal observations: {n_eligible:,}")
print(f"Baseline sensor features: {len(baseline_sensor_cols)}")

print(
    f"\nCalibration observations: "
    f"{len(baseline_calibration_check):,}"
)
print(
    f"Calibration period: "
    f"{calibration_timestamps.min()} to "
    f"{calibration_timestamps.max()}"
)

print(
    f"\nValidation observations: "
    f"{len(baseline_validation_check):,}"
)
print(
    f"Validation period: "
    f"{validation_timestamps.min()} to "
    f"{validation_timestamps.max()}"
)

print(
    "\nChronological separation verified: "
    f"{calibration_timestamps.max() < validation_timestamps.min()}"
)

print("\nRule-based empirical limits — verification copy")
print(f"Features with limits: {len(baseline_rule_limits_check)}")
display(baseline_rule_limits_check)

print("\nSimple statistical reference — verification copy")
print(
    "Sensors with zero calibration standard deviation: "
    f"{baseline_simple_reference_check['calibration_std'].eq(0).sum()}"
)
display(baseline_simple_reference_check)

NOTEBOOK 04 BASELINE PROTOCOL — VERIFICATION
Eligible normal observations: 395,298
Baseline sensor features: 51

Calibration observations: 316,238
Calibration period: 2015-12-28 10:00:00 to 2016-01-01 15:51:19

Validation observations: 79,060
Validation period: 2016-01-01 15:51:20 to 2016-01-02 14:59:59

Chronological separation verified: True

Rule-based empirical limits — verification copy
Features with limits: 51


,lower_limit,upper_limit
FIT101,0.000000,2.669827
LIT101,314.571600,814.887300
MV101,1.000000,2.000000
P101,1.000000,2.000000
P102,1.000000,1.000000
AIT201,175.948500,263.426100
AIT202,8.355102,8.692514
AIT203,291.668800,356.036900
FIT201,0.000000,2.473200
MV201,1.000000,2.000000



Simple statistical reference — verification copy
Sensors with zero calibration standard deviation: 9


,calibration_mean,calibration_std
FIT101,1.842657,1.135979
LIT101,588.903952,115.436900
MV101,1.715072,0.463117
P101,1.750536,0.432703
P102,1.000183,0.013542
AIT201,220.919175,34.202924
AIT202,8.498531,0.098962
AIT203,322.458188,12.264168
FIT201,1.838846,1.058179
MV201,1.747159,0.444177


In [38]:
# Reconstruct and audit all three frozen baseline detectors.
# Original Notebook 04 results are retained as comparison references.
# This cell does not tune models or modify saved artifacts.

import pandas as pd

# ------------------------------------------------------------
# 1. Verify the existing calibration and validation partitions.
# ------------------------------------------------------------

EXPECTED_CALIBRATION_ROWS = 316_238
EXPECTED_VALIDATION_ROWS = 79_060
EXPECTED_SENSOR_COUNT = 51

if len(baseline_calibration_check) != EXPECTED_CALIBRATION_ROWS:
    raise ValueError("The reconstructed baseline calibration size is incorrect.")

if len(baseline_validation_check) != EXPECTED_VALIDATION_ROWS:
    raise ValueError("The reconstructed baseline validation size is incorrect.")

if len(baseline_sensor_cols) != EXPECTED_SENSOR_COUNT:
    raise ValueError("The baseline sensor feature count is incorrect.")

if not baseline_calibration_check[baseline_sensor_cols].notna().all().all():
    raise ValueError("Missing values detected in baseline calibration features.")

if not baseline_validation_check[baseline_sensor_cols].notna().all().all():
    raise ValueError("Missing values detected in baseline validation features.")

validation_features = baseline_validation_check[baseline_sensor_cols]

# Recorded Notebook 04 validation results.
expected_flag_counts = {
    "Rule-Based Detection": 73_295,
    "Rolling Statistical Detection": 33_506,
    "Simple Statistical Threshold": 10_852,
}

# ------------------------------------------------------------
# 2. Rule-Based Detection
# Original limits: calibration 1st and 99th percentiles.
# Decision rule: flag if at least one sensor violates its limits.
# ------------------------------------------------------------

rule_lower = baseline_rule_limits_check["lower_limit"]
rule_upper = baseline_rule_limits_check["upper_limit"]

rule_violations_check = (
    validation_features.lt(rule_lower, axis="columns")
    | validation_features.gt(rule_upper, axis="columns")
)

rule_scores_check = rule_violations_check.sum(axis=1)
rule_flags_check = rule_scores_check.gt(0)

# ------------------------------------------------------------
# 3. Rolling Statistical Detection
# Documented specification:
# - 60 preceding observations; current observation excluded.
# - Sample standard deviation (ddof=1).
# - Zero rolling standard deviation is treated as undefined.
# - Flag if at least one defined absolute Z-score exceeds 3.0.
#
# The original calculation cell is unavailable. This is an
# independent reconstruction of the documented specification,
# not a claim of line-by-line equivalence with the original code.
# ------------------------------------------------------------

ROLLING_WINDOW = 60
ROLLING_Z_THRESHOLD = 3.0

eligible_timestamps = pd.to_datetime(
    eligible_normal["Timestamp"],
    errors="coerce",
)

if eligible_timestamps.isna().any():
    raise ValueError("Invalid timestamps found in eligible_normal.")

if not eligible_timestamps.is_monotonic_increasing:
    raise ValueError(
        "eligible_normal must be chronologically ordered for the "
        "rolling reconstruction."
    )

calibration_end = len(baseline_calibration_check)

# Retain the 60 observations immediately before validation so
# its first observation has valid historical context.
history_start = calibration_end - ROLLING_WINDOW

if history_start < 0:
    raise ValueError("Insufficient historical observations for rolling scores.")

rolling_input = eligible_normal.iloc[history_start:][baseline_sensor_cols]

rolling_means = (
    rolling_input
    .rolling(window=ROLLING_WINDOW, min_periods=ROLLING_WINDOW)
    .mean()
    .shift(1)
)

rolling_stds = (
    rolling_input
    .rolling(window=ROLLING_WINDOW, min_periods=ROLLING_WINDOW)
    .std(ddof=1)
    .shift(1)
)

# Mask undefined zero-variance rolling windows.
rolling_stds = rolling_stds.mask(rolling_stds.eq(0))

# The first 60 rows are historical context; score validation only.
validation_offset = ROLLING_WINDOW

rolling_validation_values = rolling_input.iloc[validation_offset:]
rolling_validation_means = rolling_means.iloc[validation_offset:]
rolling_validation_stds = rolling_stds.iloc[validation_offset:]

if len(rolling_validation_values) != EXPECTED_VALIDATION_ROWS:
    raise AssertionError("Rolling scores do not align with validation rows.")

if not rolling_validation_values.index.equals(validation_features.index):
    raise AssertionError(
        "Rolling validation observations do not align with baseline validation."
    )

validation_rolling_z_scores_check = (
    rolling_validation_values
    .sub(rolling_validation_means)
    .div(rolling_validation_stds)
    .abs()
)

rolling_violations_check = (
    validation_rolling_z_scores_check
    .gt(ROLLING_Z_THRESHOLD)
    .fillna(False)
)

rolling_scores_check = rolling_violations_check.sum(axis=1)
rolling_flags_check = rolling_scores_check.gt(0)

# ------------------------------------------------------------
# 4. Simple Statistical Threshold
# Original reference: calibration mean and sample standard deviation.
# Decision rule: flag if at least one scorable sensor has |z| > 3.
# Zero-standard-deviation sensors remain masked.
# ------------------------------------------------------------

simple_means = baseline_simple_reference_check["calibration_mean"]
simple_stds = baseline_simple_reference_check["calibration_std"]

valid_simple_stds = simple_stds.mask(simple_stds.eq(0))

simple_z_scores_check = (
    validation_features
    .sub(simple_means, axis="columns")
    .div(valid_simple_stds, axis="columns")
)

simple_violations_check = (
    simple_z_scores_check
    .abs()
    .gt(3.0)
    .fillna(False)
)

simple_scores_check = simple_violations_check.sum(axis=1)
simple_flags_check = simple_scores_check.gt(0)

# ------------------------------------------------------------
# 5. Compare reconstructed counts with Notebook 04.
# ------------------------------------------------------------

reconstructed_flags = {
    "Rule-Based Detection": rule_flags_check,
    "Rolling Statistical Detection": rolling_flags_check,
    "Simple Statistical Threshold": simple_flags_check,
}

comparison_rows = []

for method, flags in reconstructed_flags.items():
    recorded_count = expected_flag_counts[method]
    reconstructed_count = int(flags.sum())

    comparison_rows.append({
        "method": method,
        "validation_observations": len(flags),
        "recorded_flagged_observations": recorded_count,
        "reconstructed_flagged_observations": reconstructed_count,
        "counts_match": reconstructed_count == recorded_count,
        "reconstructed_flag_rate_pct": (
            100 * reconstructed_count / len(flags)
        ),
    })

baseline_audit_comparison = pd.DataFrame(comparison_rows)

print("FROZEN BASELINE RECONSTRUCTION AUDIT")
print("=" * 65)
display(baseline_audit_comparison)

# ------------------------------------------------------------
# 6. Review supporting diagnostics, including rolling missingness.
# ------------------------------------------------------------

print("\nRule-Based Detection")
print(f"Mean sensor violations: {rule_scores_check.mean():.4f}")
print(f"Maximum sensor violations: {rule_scores_check.max()}")

print("\nRolling Statistical Detection")
print(f"Mean sensor violations: {rolling_scores_check.mean():.4f}")
print(f"Maximum sensor violations: {rolling_scores_check.max()}")
print(
    "Mean scorable sensors per observation: "
    f"{validation_rolling_z_scores_check.notna().sum(axis=1).mean():.4f}"
)
print(
    "Missing rolling Z-scores: "
    f"{int(validation_rolling_z_scores_check.isna().sum().sum()):,}"
)
print(
    "Sensors with no defined rolling Z-scores in validation: "
    f"{int((~validation_rolling_z_scores_check.notna().any(axis=0)).sum())}"
)

print("\nSimple Statistical Threshold")
print(f"Mean sensor violations: {simple_scores_check.mean():.4f}")
print(f"Maximum sensor violations: {simple_scores_check.max()}")
print(
    "Sensors with zero calibration standard deviation: "
    f"{int(simple_stds.eq(0).sum())}"
)

print("\nMost frequently violated rule-based limits:")
display(
    rule_violations_check.sum(axis=0)
    .sort_values(ascending=False)
    .head(10)
    .rename("violation_count")
    .to_frame()
)

print("\nMost frequently exceeded simple statistical thresholds:")
display(
    simple_violations_check.sum(axis=0)
    .sort_values(ascending=False)
    .head(10)
    .rename("violation_count")
    .to_frame()
)

print("\nAudit outcome")
if baseline_audit_comparison["counts_match"].all():
    print(
        "All three reconstructed flagged-observation counts match "
        "their recorded Notebook 04 results. Review supporting "
        "diagnostics before declaring the rolling reconstruction equivalent."
    )
else:
    print(
        "One or more reconstructed counts differ from Notebook 04. "
        "Preserve the original results and investigate the discrepancy; "
        "do not change baseline definitions to force agreement."
    )

FROZEN BASELINE RECONSTRUCTION AUDIT


,method,validation_observations,recorded_flagged_observations,reconstructed_flagged_observations,counts_match,reconstructed_flag_rate_pct
0,Rule-Based Detection,79060,73295,73295,True,92.708070
1,Rolling Statistical Detection,79060,33506,32672,False,41.325576
2,Simple Statistical Threshold,79060,10852,10852,True,13.726284



Rule-Based Detection
Mean sensor violations: 1.9914
Maximum sensor violations: 16

Rolling Statistical Detection
Mean sensor violations: 0.6042
Maximum sensor violations: 24
Mean scorable sensors per observation: 21.9822
Missing rolling Z-scores: 2,294,150
Sensors with no defined rolling Z-scores in validation: 11

Simple Statistical Threshold
Mean sensor violations: 0.2273
Maximum sensor violations: 13
Sensors with zero calibration standard deviation: 9

Most frequently violated rule-based limits:


,violation_count
AIT201,63020
AIT402,21867
AIT203,13506
AIT202,11921
AIT504,10927
AIT502,10043
DPIT301,4923
FIT201,3231
LIT301,3176
FIT504,2986



Most frequently exceeded simple statistical thresholds:


,violation_count
AIT401,4410
MV304,3722
MV303,3318
MV301,1413
FIT601,1037
P602,921
MV302,593
MV101,453
MV201,411
AIT502,311



Audit outcome
One or more reconstructed counts differ from Notebook 04. Preserve the original results and investigate the discrepancy; do not change baseline definitions to force agreement.


In [39]:
# Locate the project root from the current working directory.
CURRENT_DIR = Path("/content/drive/MyDrive")

PROJECT_ROOT = Path("/content/drive/MyDrive")

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'models/attack_evaluation_sequences.npz' and 'data/raw/attack.csv'."
    )

ATTACK_PATH = PROJECT_ROOT / "swat-dataset" / "attack.csv"
SEQUENCE_PATH = PROJECT_ROOT / "swat-artifacts" / "attack_evaluation_sequences.npz"


# Read only timestamps and ground-truth labels from the source file.
source_header = pd.read_csv(ATTACK_PATH, nrows=0)
column_lookup = {str(column).strip(): column for column in source_header.columns}

required_columns = {"Timestamp", "Normal/Attack"}
missing_columns = required_columns.difference(column_lookup)

if missing_columns:
    raise KeyError(
        f"Required source columns are missing: {sorted(missing_columns)}"
    )

attack_metadata = pd.read_csv(
    ATTACK_PATH,
    usecols=[
        column_lookup["Timestamp"],
        column_lookup["Normal/Attack"],
    ],
).rename(
    columns={
        column_lookup["Timestamp"]: "Timestamp",
        column_lookup["Normal/Attack"]: "Label",
    }
)

attack_metadata["Timestamp"] = pd.to_datetime(
    attack_metadata["Timestamp"],
    errors="coerce",
)

attack_metadata["Label"] = (
    attack_metadata["Label"].astype("string").str.strip().str.casefold()
)


# Inspect the saved sequence artifact and its endpoint indices.
with np.load(SEQUENCE_PATH, allow_pickle=False) as artifact:
    if "attack_sequences" in globals():
        attack_sequence_array = attack_sequences
        sequence_source = "Existing in-memory attack_sequences array"
    else:
        attack_sequence_array = artifact["attack_sequences"]
        sequence_source = "Saved attack_evaluation_sequences.npz artifact"

    endpoint_arrays = {
        key: np.asarray(artifact[key])
        for key in artifact.files
        if key != "attack_sequences"
    }

if attack_sequence_array.ndim != 3:
    raise AssertionError(
        f"Expected a three-dimensional sequence array; "
        f"received shape {attack_sequence_array.shape}."
    )

n_attack_rows = len(attack_metadata)
n_sequences, sequence_length, n_features = attack_sequence_array.shape

if sequence_length != 30:
    raise AssertionError(
        f"Expected a sequence length of 30; received {sequence_length}."
    )

if n_features != 42:
    raise AssertionError(
        f"Expected 42 model features; received {n_features}."
    )

expected_sequence_count = n_attack_rows - sequence_length + 1

if n_sequences != expected_sequence_count:
    raise AssertionError(
        f"Sequence count mismatch: expected {expected_sequence_count:,}, "
        f"received {n_sequences:,}."
    )

expected_endpoint_indices = np.arange(
    sequence_length - 1,
    n_attack_rows,
    dtype=np.int64,
)

matching_endpoint_keys = [
    key
    for key, values in endpoint_arrays.items()
    if values.ndim == 1
    and len(values) == n_sequences
    and np.issubdtype(values.dtype, np.integer)
    and np.array_equal(values, expected_endpoint_indices)
]

if len(matching_endpoint_keys) != 1:
    raise AssertionError(
        "Could not uniquely verify the saved sequence endpoint indices. "
        f"Artifact keys: {list(endpoint_arrays)}; "
        f"matching endpoint keys: {matching_endpoint_keys}."
    )

endpoint_indices = endpoint_arrays[matching_endpoint_keys[0]]


# Verify source timestamps, chronological order, and endpoint labels.
if attack_metadata["Timestamp"].isna().any():
    raise AssertionError("The attack source contains invalid or missing timestamps.")

if not attack_metadata["Timestamp"].is_monotonic_increasing:
    raise AssertionError(
        "Attack source timestamps are not in chronological order."
    )

if not attack_metadata["Label"].eq("attack").all():
    raise AssertionError(
        "The attack source contains missing, unexpected, or non-attack labels."
    )

endpoint_timestamps = attack_metadata["Timestamp"].iloc[endpoint_indices]
endpoint_labels = attack_metadata["Label"].iloc[endpoint_indices]

if endpoint_timestamps.isna().any():
    raise AssertionError("One or more sequence endpoint timestamps are missing.")

if not endpoint_labels.eq("attack").all():
    raise AssertionError(
        "One or more sequence endpoints do not have the expected attack label."
    )


# Report the verified alignment evidence.
print("ATTACK SEQUENCE ALIGNMENT AUDIT")
print("=" * 38)
print(f"Sequence source: {sequence_source}")
print(f"Source attack rows: {n_attack_rows:,}")
print(f"Saved attack sequences: {n_sequences:,}")
print(f"Sequence length: {sequence_length}")
print(f"Model features: {n_features}")
print(f"Endpoint-index artifact key: {matching_endpoint_keys[0]}")
print(f"Endpoint indices match the expected convention: True")
print(f"Source timestamps are valid and chronological: True")
print(f"All source labels are Attack: True")
print(f"All sequence endpoint labels are Attack: True")
print(f"First sequence endpoint: {endpoint_timestamps.iloc[0]}")
print(f"Last sequence endpoint: {endpoint_timestamps.iloc[-1]}")

ATTACK SEQUENCE ALIGNMENT AUDIT
Sequence source: Existing in-memory attack_sequences array
Source attack rows: 54,621
Saved attack sequences: 54,592
Sequence length: 30
Model features: 42
Endpoint-index artifact key: attack_end_indices
Endpoint indices match the expected convention: True
Source timestamps are valid and chronological: True
All source labels are Attack: True
All sequence endpoint labels are Attack: True
First sequence endpoint: 2015-12-28 10:29:43
Last sequence endpoint: 2016-01-02 13:41:11


In [40]:
# Locate the project root.
CURRENT_DIR = Path("/content/drive/MyDrive")

PROJECT_ROOT = Path("/content/drive/MyDrive")

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'models/attack_evaluation_sequences.npz'."
    )

MODELS_DIR = PROJECT_ROOT / "swat-artifacts"

# Establish the expected number of attack sequence endpoints.
if "attack_sequences" in globals():
    n_attack_sequences = len(attack_sequences)
else:
    with np.load(
        MODELS_DIR / "attack_evaluation_sequences.npz",
        allow_pickle=False,
    ) as artifact:
        n_attack_sequences = len(artifact["attack_sequences"])

score_artifact_names = [
    "traditional_ml_scores.npz",
    "lstm_autoencoder_scores.npz",
    "tcn_autoencoder_scores.npz",
    "usad_preliminary_scores.npz",
]

score_keywords = (
    "score",
    "error",
    "energy",
    "anomaly",
    "reconstruction",
    "loss",
)

print("MODEL SCORE ARTIFACT INVENTORY")
print("=" * 42)
print(f"Expected attack sequence scores: {n_attack_sequences:,}")
print()

# Inspect the established score artifacts without modifying them.
for artifact_name in score_artifact_names:
    artifact_path = MODELS_DIR / artifact_name

    print(f"Artifact: {artifact_name}")

    if not artifact_path.is_file():
        print("  Status: File not found")
        print()
        continue

    with np.load(artifact_path, allow_pickle=False) as artifact:
        if not artifact.files:
            print("  Status: No arrays found")
        else:
            for key in artifact.files:
                values = artifact[key]
                is_score_field = any(
                    keyword in key.casefold()
                    for keyword in score_keywords
                )
                matches_attack_length = (
                    values.ndim >= 1
                    and values.shape[0] == n_attack_sequences
                )

                if is_score_field or matches_attack_length:
                    print(
                        f"  Array: {key} | "
                        f"Shape: {values.shape} | "
                        f"Dtype: {values.dtype} | "
                        f"Attack-length match: {matches_attack_length}"
                    )

    print()

# Inspect relevant score arrays already present in the notebook session.
print("IN-MEMORY SCORE ARRAY INVENTORY")
print("=" * 42)

memory_candidates = []

for variable_name, value in list(globals().items()):
    if not any(
        keyword in variable_name.casefold()
        for keyword in score_keywords
    ):
        continue

    if isinstance(value, np.ndarray):
        if value.ndim >= 1 and value.shape[0] == n_attack_sequences:
            memory_candidates.append(
                (variable_name, value.shape, value.dtype)
            )

if memory_candidates:
    for variable_name, shape, dtype in sorted(memory_candidates):
        print(
            f"  Variable: {variable_name} | "
            f"Shape: {shape} | "
            f"Dtype: {dtype} | "
            f"Attack-length match: True"
        )
else:
    print("No matching in-memory score arrays were identified.")

MODEL SCORE ARTIFACT INVENTORY
Expected attack sequence scores: 54,592

Artifact: traditional_ml_scores.npz
  Array: if_threshold_scores | Shape: (47437,) | Dtype: float64 | Attack-length match: False
  Array: if_validation_scores | Shape: (79060,) | Dtype: float64 | Attack-length match: False
  Array: ocsvm_threshold_scores | Shape: (47437,) | Dtype: float64 | Attack-length match: False
  Array: ocsvm_validation_scores | Shape: (79060,) | Dtype: float64 | Attack-length match: False
  Array: lof_threshold_scores | Shape: (47437,) | Dtype: float32 | Attack-length match: False
  Array: lof_validation_scores | Shape: (79060,) | Dtype: float32 | Attack-length match: False

Artifact: lstm_autoencoder_scores.npz
  Array: threshold_scores | Shape: (47408,) | Dtype: float64 | Attack-length match: False
  Array: validation_scores | Shape: (79031,) | Dtype: float64 | Attack-length match: False

Artifact: tcn_autoencoder_scores.npz
  Array: threshold_scores | Shape: (47408,) | Dtype: float64 | At

In [ ]:
# Resolve the project root.
CURRENT_DIR = Path("/content/drive/MyDrive")

PROJECT_ROOT = Path("/content/drive/MyDrive")

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing the saved scaler "
        "and data/raw/attack.csv."
    )

MODELS_DIR = PROJECT_ROOT / "swat-artifacts"
ATTACK_PATH = PROJECT_ROOT / "swat-dataset" / "attack.csv"


# Load the existing training-fitted scaler and frozen traditional models.
scaler = joblib.load(MODELS_DIR / "scaler.joblib")

traditional_models = {
    "if": joblib.load(MODELS_DIR / "isolation_forest.joblib"),
    "ocsvm": joblib.load(MODELS_DIR / "sgd_one_class_svm.joblib"),
    "lof": joblib.load(MODELS_DIR / "lof_model.joblib"),
}


# Establish the exact feature order used during scaler fitting.
if hasattr(scaler, "feature_names_in_"):
    model_feature_cols = [
        str(column) for column in scaler.feature_names_in_
    ]
elif "feature_cols" in globals():
    model_feature_cols = list(feature_cols)
else:
    raise RuntimeError(
        "The saved scaler does not expose feature names and the notebook "
        "does not contain feature_cols. The required feature order "
        "must be established before scoring."
    )

if len(model_feature_cols) != 42:
    raise AssertionError(
        f"Expected 42 model features; found {len(model_feature_cols)}."
    )

if len(set(model_feature_cols)) != len(model_feature_cols):
    raise AssertionError("The model feature list contains duplicate names.")

if getattr(scaler, "n_features_in_", None) != len(model_feature_cols):
    raise AssertionError(
        "The scaler feature count does not match the established feature list."
    )


# Resolve source column names without changing the raw file.
source_header = pd.read_csv(ATTACK_PATH, nrows=0)
column_lookup = {
    str(column).strip(): column
    for column in source_header.columns
}

required_columns = [
    "Timestamp",
    "Normal/Attack",
    *model_feature_cols,
]

missing_columns = [
    column for column in required_columns
    if column not in column_lookup
]

if missing_columns:
    raise KeyError(
        f"Required columns are missing from attack.csv: {missing_columns}"
    )

source_columns = [column_lookup[column] for column in required_columns]

attack_frame = pd.read_csv(
    ATTACK_PATH,
    usecols=source_columns,
).rename(
    columns={
        column_lookup[column]: column
        for column in required_columns
    }
)

attack_frame["Timestamp"] = pd.to_datetime(
    attack_frame["Timestamp"],
    errors="coerce",
)

attack_frame["Normal/Attack"] = (
    attack_frame["Normal/Attack"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

if attack_frame["Timestamp"].isna().any():
    raise AssertionError("Attack data contains invalid timestamps.")

if not attack_frame["Timestamp"].is_monotonic_increasing:
    raise AssertionError("Attack timestamps are not in chronological order.")

if not attack_frame["Normal/Attack"].eq("attack").all():
    raise AssertionError(
        "The attack file contains unexpected or missing ground-truth labels."
    )


# Validate and transform the original feature values using the saved scaler.
attack_features = attack_frame[model_feature_cols].apply(
    pd.to_numeric,
    errors="coerce",
)

if not np.isfinite(attack_features.to_numpy(dtype=np.float64)).all():
    raise AssertionError(
        "Attack features contain missing or non-finite values. "
        "No imputation has been applied."
    )

scaled_attack_values = scaler.transform(attack_features)

scaled_attack_features = pd.DataFrame(
    scaled_attack_values,
    columns=model_feature_cols,
)

if not np.isfinite(
    scaled_attack_features.to_numpy(dtype=np.float64)
).all():
    raise AssertionError("Scaled attack features contain non-finite values.")


# Apply the established score convention: higher scores indicate greater
# anomaly evidence. No model fitting or threshold selection occurs here.
attack_scores = {}

for model_name, model in traditional_models.items():
    if not callable(getattr(model, "score_samples", None)):
        raise TypeError(
            f"The saved {model_name} model does not expose score_samples()."
        )

    if model_name == "lof" and not getattr(model, "novelty", False):
        raise AssertionError(
            "The saved LOF model is not configured for novelty detection."
        )

    model_feature_names = getattr(model, "feature_names_in_", None)

    if model_feature_names is not None:
        if list(model_feature_names) != model_feature_cols:
            raise AssertionError(
                f"The {model_name} model feature order differs from the "
                "scaler feature order."
            )

    scores = -np.asarray(
        model.score_samples(scaled_attack_features),
        dtype=np.float64,
    )

    if scores.shape != (len(attack_frame),):
        raise AssertionError(
            f"{model_name} returned shape {scores.shape}; "
            f"expected {(len(attack_frame),)}."
        )

    if not np.isfinite(scores).all():
        raise AssertionError(
            f"{model_name} produced non-finite attack scores."
        )

    attack_scores[f"{model_name}_attack_scores"] = scores


# Save attack scores separately from the existing normal-data score artifact.
OUTPUT_PATH = MODELS_DIR / "attack_traditional_ml_scores.npz"

np.savez_compressed(
    OUTPUT_PATH,
    **attack_scores,
    attack_row_indices=np.arange(len(attack_frame), dtype=np.int64),
    attack_timestamps=attack_frame["Timestamp"].to_numpy(
        dtype="datetime64[ns]"
    ),
    attack_labels=attack_frame["Normal/Attack"].to_numpy(dtype="U"),
)

print("TRADITIONAL ML ATTACK SCORING")
print("=" * 38)
print(f"Attack observations: {len(attack_frame):,}")
print(f"Model features: {len(model_feature_cols)}")
print("Scaler: Existing training-fitted scaler")
print("Model fitting or tuning on attack data: False")
print("Threshold calibration using attack data: False")

for array_name, scores in attack_scores.items():
    print(
        f"{array_name}: shape={scores.shape}, "
        f"minimum={scores.min():.6f}, "
        f"median={np.median(scores):.6f}, "
        f"maximum={scores.max():.6f}"
    )

# print(f"Saved artifact: {OUTPUT_PATH}")

In [ ]:
# Locate the project root.
CURRENT_DIR = Path("/content/drive/MyDrive")

PROJECT_ROOT = Path("/content/drive/MyDrive")

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate models/attack_traditional_ml_scores.npz."
    )

score_path = PROJECT_ROOT / "swat-artifacts" / "attack_traditional_ml_scores.npz"

with np.load(score_path, allow_pickle=False) as artifact:
    required_keys = {
        "if_attack_scores",
        "ocsvm_attack_scores",
        "lof_attack_scores",
        "attack_row_indices",
        "attack_timestamps",
        "attack_labels",
    }

    missing_keys = required_keys.difference(artifact.files)

    if missing_keys:
        raise KeyError(
            f"Attack score artifact is missing keys: {sorted(missing_keys)}"
        )

    saved = {key: artifact[key].copy() for key in required_keys}


n_rows = len(saved["attack_row_indices"])
expected_indices = np.arange(n_rows, dtype=np.int64)

if n_rows != 54_621:
    raise AssertionError(f"Expected 54,621 attack rows; found {n_rows:,}.")

if not np.array_equal(saved["attack_row_indices"], expected_indices):
    raise AssertionError("Saved attack row indices are not aligned.")

if not np.all(saved["attack_labels"] == "attack"):
    raise AssertionError("Saved ground-truth labels are not all Attack.")

for score_name in (
    "if_attack_scores",
    "ocsvm_attack_scores",
    "lof_attack_scores",
):
    scores = saved[score_name]

    if scores.shape != (n_rows,):
        raise AssertionError(
            f"{score_name} has shape {scores.shape}; expected {(n_rows,)}."
        )

    if not np.isfinite(scores).all():
        raise AssertionError(f"{score_name} contains non-finite values.")


# Compare saved timestamps against the raw source timestamps.
source_header = pd.read_csv(
    PROJECT_ROOT / "swat-dataset" / "attack.csv",
    nrows=0,
)
column_lookup = {str(column).strip(): column for column in source_header.columns}

source_timestamps = pd.read_csv(
    PROJECT_ROOT / "swat-dataset" / "attack.csv",
    usecols=[column_lookup["Timestamp"]],
)[column_lookup["Timestamp"]]

source_timestamps = pd.to_datetime(
    source_timestamps,
    errors="coerce",
).to_numpy(dtype="datetime64[ns]")

if not np.array_equal(saved["attack_timestamps"], source_timestamps):
    raise AssertionError(
        "Saved attack timestamps do not match the raw source row by row."
    )

print("TRADITIONAL ATTACK SCORE ALIGNMENT")
print("=" * 40)
print(f"Attack rows: {n_rows:,}")
print("Row indices aligned: True")
print("Timestamps match raw source row by row: True")
print("All ground-truth labels are Attack: True")
print("All three score arrays have the expected length: True")
print("All three score arrays contain finite values: True")
print("Audit result: PASS")

TRADITIONAL ATTACK SCORE ALIGNMENT
Attack rows: 54,621
Row indices aligned: True
Timestamps match raw source row by row: True
All ground-truth labels are Attack: True
All three score arrays have the expected length: True
All three score arrays contain finite values: True
Audit result: PASS
